# Japan 14-Day Itinerary Planner with Gemini

This notebook demonstrates how to use the Gemini API with Google Search Grounding to generate a highly detailed, 14-day itinerary for a young couple visiting Japan in September. By leveraging grounding, Gemini can fetch real-time information, including specific places, routes, and cultural hotspots that are currently relevant.

In [7]:
#!pip install google-genai IPython

In [41]:
import os
import json
from google import genai
from google.genai import types
from IPython.display import Markdown, display

# Load configuration from config.json
try:
    with open('../config.json', 'r') as f:
        config_data = json.load(f)
        PROJECT_ID = config_data.get('gcp', {}).get('projectId', 'YOUR_PROJECT_ID')
        MODEL_ID = config_data.get('models', {}).get('gemini', {}).get('activeModelId', 'gemini-3.1-pro-preview')
except Exception as e:
    print(f"Could not read config.json, using defaults. Error: {e}")
    PROJECT_ID = "vertex-ai-382806"
    MODEL_ID = "gemini-3.5-flash"

# You can also load region if it was in the config, but we will default to global
LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION", "global") 

try:
    client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)
    print("Vertex AI Gemini Client initialized successfully.")
    print(f"Using Project ID: {PROJECT_ID}")
    print(f"Using Model ID: {MODEL_ID}")
except Exception as e:
    print(f"Error initializing client: {e}")
    print("Please ensure your Google Cloud credentials, project ID, and location are set correctly.")

Vertex AI Gemini Client initialized successfully.
Using Project ID: vertex-ai-382806
Using Model ID: gemini-3.5-flash


In [9]:
# Read the prompt and preferences from external files
prompt_file = "prompts/japan.md"
prefs_file = "preferences.md"

# Fallback paths just in case the notebook is run from the root directory instead of /notebooks
if not os.path.exists(prompt_file):
    prompt_file = "notebooks/prompts/japan.md"
if not os.path.exists(prefs_file):
    prefs_file = "notebooks/preferences.md"

try:
    with open(prompt_file, "r") as f:
        base_prompt = f.read().strip()
except FileNotFoundError:
    base_prompt = "Please create a highly curated list of places to go in Japan."

try:
    with open(prefs_file, "r") as f:
        preferences = f.read().strip()
except FileNotFoundError:
    preferences = "couple trip\nyoung\nmid luxury budget\nlove adventure\nnature\nhistory"

# Combine both into the final prompt variable
prompt = f"{base_prompt}\n\n### User Preferences:\n{preferences}"

# Configure the generation to use Google Search Grounding.
# This gives the model access to up-to-date Google Search and Google Maps data.
config = types.GenerateContentConfig(
    tools=[{"google_search": {}}],
    temperature=0.7,
)

In [10]:
print(f"Generating itinerary with {MODEL_ID}... This might take a few moments as it fetches real-time data.")
try:
    # Use the model defined in config.json
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=config
    )
    
    # Display the result formatted as Markdown
    display(Markdown(response.text))
    
    # Save the output to a file
    output_dir = "outputs"
    if not os.path.exists(output_dir) and os.path.exists("notebooks/outputs"):
        output_dir = "notebooks/outputs"
    os.makedirs(output_dir, exist_ok=True)
    
    output_file = os.path.join(output_dir, "japan_itinerary.md")
    with open(output_file, "w") as f:
        f.write(response.text)
    print(f"\nSaved itinerary to {output_file}")
    
    # Optional: Display the search grounding sources if available
    if response.candidates and response.candidates[0].grounding_metadata:
        print("\n--- Grounding Sources ---")
        metadata = response.candidates[0].grounding_metadata
        if hasattr(metadata, 'grounding_chunks'):
            for chunk in metadata.grounding_chunks:
                if hasattr(chunk, 'web') and chunk.web:
                    print(f"- {chunk.web.title}: {chunk.web.uri}")
except Exception as e:
    print(f"An error occurred during generation: {e}")

Generating itinerary with gemini-3.5-flash... This might take a few moments as it fetches real-time data.


Welcome to Japan in September! This is a magical, transitional month to visit. The intense summer heat begins to fade, yielding to crisp mountain breezes and the very first hints of autumn foliage in the north. It is the perfect time for an active, culturally curious young couple to explore. 

With a **mid-luxury budget**, you can comfortably blend high-end adventure, boutique stays (like premium *ryokans* and eco-resorts), and deeply rich historical experiences.

---

### 1. Elevated Classic Hubs (With an Adventurous & Romantic Twist)
Even Japan's most famous cities have hidden, high-end, and adventurous angles if you know where to look.

#### **Tokyo (The Modern & Historic Spark)**
*   **The Vibe:** High-energy culture, cutting-edge art, and hidden historic pockets.
*   **Mid-Luxury Stay:** Stay at a boutique design hotel like **The Tokyo Edition, Toranomon** or experience a high-end, modern ryokan at **Hoshinoya Tokyo**.
*   **Adventure & Nature:** Take a day trip to the **Okutama** region or **Mount Mitake** (on the western edge of Tokyo) for river rafting, canyoning, and hiking through ancient shrines. 
*   **Cultural Hotspot:** Attend the **Tokyo Grand Sumo Tournament** (held annually in mid-to-late September at the Ryogoku Kokugikan). Snag premium box seats for an upscale, authentic experience.
*   **September Highlight:** Walk through **Shinjuku Gyoen National Garden** to catch the very first golden leaves, then head to a high-end rooftop bar in Shibuya for sunset cocktails.

#### **Kyoto (Timeless Romance & Hidden Trails)**
*   **The Vibe:** Ancient temples, bamboo groves, and exquisite dining.
*   **Mid-Luxury Stay:** Book **Sowaka**, a stunning luxury ryokan in the historic Gion district, or **Hoshinoya Kyoto**, accessed via a private wooden boat ride up the Oi River.
*   **Adventure & History:** Skip the crowded Fushimi Inari at midday. Instead, hike the mystical **Kurama to Kibune trail**—a mountain path connecting two ancient shrines. Finish the hike with *kawadoko* dining (eating on wooden platforms suspended directly over a rushing river, a refreshing late-summer tradition available through September).
*   **September Highlight:** Rent a private wooden rowboat on the Hozu River in Arashiyama to enjoy the cooler breeze and early autumn colors creeping onto the hillsides.

---

### 2. Alpine Nature & Adventure Hotspots
September is prime hiking season in the Japanese Alps—the air is crisp, the skies are clear, and the summer crowds have dispersed.

#### **Kamikochi Valley & Matsumoto (Nagano)**
*   **The Vibe:** Dramatic alpine peaks, crystal-clear rivers, and samurai history.
*   **The Adventure:** **Kamikochi** is a pristine, car-free highland valley in the Chubu Sangaku National Park. Hike along the Azusa River to Taisho Pond, which perfectly reflects the active volcano, Mount Yakedake. 
*   **The History:** Base yourselves in Matsumoto to explore **Matsumoto Castle** (one of Japan's most complete and beautiful original 16th-century "Crow Castles").
*   **Mid-Luxury Stay:** Stay at **Myojinkan**, a secluded luxury hot-spring ryokan nestled deep in the mountains of Matsumoto, offering private outdoor *onsen* (hot spring) baths.

#### **Daisetsuzan National Park (Hokkaido)**
*   **The Vibe:** Wild, untamed volcanic landscapes and the absolute first autumn colors in Japan.
*   **The Adventure:** If you visit in mid-to-late September, Daisetsuzan is the first place in the country where the leaves turn fiery red and brilliant gold. Take the ropeway up **Mount Asahidake** (an active volcano) and hike through steaming volcanic vents and alpine lakes.
*   **Relaxation:** After a long hike, soak in the sulfurous waters of **Sounkyo Onsen**. Stay at a premium mountain lodge like **La Vista Daisetsuzan**.

---

### 3. Off-The-Beaten-Path Hidden Gems
For an adventurous couple looking to escape the typical tourist trail, these highly-rated hidden gems offer deep nature, romance, and historic mystery.

#### **The Iya Valley (Shikoku)**
*   **The Vibe:** A mystical, misty river gorge known as the "Tibet of Japan." Historically, defeated samurai clans hid in these steep ravines during the 12th century.
*   **The Adventure:** Cross the famous **Kazurabashi** (bridges made of living mountain vines), go white-water rafting on the roaring Yoshino River, or rent a car to drive the winding, narrow mountain roads.
*   **Mid-Luxury Stay:** Stay at **Chiiori**, a beautifully restored 300-year-old thatched-roof farmhouse. Preserved by Japanologist Alex Kerr, it features a traditional *irori* (floor hearth) blackened by centuries of smoke, but is retrofitted with modern luxury amenities like underfloor heating and a gorgeous wooden soaking tub.

#### **Yakushima Island (Kyushu)**
*   **The Vibe:** A subtropical, ancient island that inspired the mystical mossy forests of Studio Ghibli’s *Princess Mononoke*.
*   **The Adventure:** Hike through forests of thousand-year-old *Yakusugi* (ancient cedar trees), swim under towering waterfalls like **Oko-no-taki**, and kayak down pristine coastal rivers.
*   **Mid-Luxury Stay:** Stay at the **Sankara Hotel & Spa Yakushima**. This stunning, auberge-style luxury resort is nestled between the mountains and the sea. It features high-end villas, an infinity pool, and a fine-dining French restaurant utilizing fresh, hyper-local Yakushima ingredients.

#### **Kiso Valley & The Nakasendo Trail (Nagano/Gifu)**
*   **The Vibe:** Stepping directly back into the Edo period.
*   **The Adventure & History:** Hike the historic **Nakasendo Trail** (an ancient post route) between the beautifully preserved towns of **Magome** and **Tsumago**. The 8km trail winds through lush forests, bamboo groves, and past waterfalls.
*   **Relaxation:** Stay in a boutique ryokan in the nearby Kiso Valley, enjoying local sake, mountain-vegetable cuisine, and private cypress baths.

---

### 4. September-Exclusive Cultural Hotspots & Festivals
September in Japan hosts some of the most vibrant, high-energy festivals and natural spectacles of the year.

#### **Kishiwada Danjiri Matsuri (Osaka)**
*   **What it is:** One of Japan’s wildest and most thrilling traditional festivals.
*   **The Vibe:** Teams from different neighborhoods haul massive, 4-ton wooden floats (*danjiri*) through the streets at breakneck speeds, making terrifying 90-degree turns on narrow corners.
*   **When:** Mid-September.
*   **Couple Tip:** Secure paid grandstand seating at the *Cancan Bayside Mall* for a comfortable, unobstructed view of the action, then stroll the street food stalls hand-in-hand to try local Osaka specialties like *takoyaki* and *okonomiyaki*.

#### **Kinchakuda Manjushage Park (Saitama)**
*   **What it is:** The largest field of *higanbana* (Red Spider Lilies) in Japan.
*   **The Vibe:** In late September, over 5 million brilliant red spider lilies bloom simultaneously under the shade of trees along the looping Koma River, creating an otherworldly, romantic crimson carpet.
*   **Couple Tip:** It’s an easy 75-minute day trip from Tokyo. Go early on a weekday morning to beat the crowds, grab some local chestnut treats at the festival stalls, and take breathtaking, vibrant couple photos.

---

### Practical Planning Advice for a September Honeymoon/Couple Trip
1.  **Weather & Packing:** September is a transitional month. Early September can still feel like late summer (warm and humid), while late September brings cool, pleasant autumn breezes. Pack light layers, comfortable walking shoes, and sturdy hiking boots for Yakushima or Kamikochi.
2.  **Typhoon Season:** September is peak typhoon season in Japan. While major disruptions are rare, it is highly recommended to purchase comprehensive travel insurance and keep your travel plans slightly flexible.
3.  **Getting Around:** For this mix of remote hidden gems (like Iya Valley and Yakushima) and major cities, a combination of the Shinkansen (bullet train) for the main routes and renting a car for the rural segments (Shikoku/Nagano) is the ultimate way to travel in style and comfort.


Saved itinerary to outputs/japan_itinerary.md

--- Grounding Sources ---
- rakuten.com: https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHUY5J6RG8ei1GLp3Kag7tKMmLHmBfnSNlCi1E7Kf0aQvgj-zMZySd3H2Kh1NTBtB-qmeMu6b51Zfy3sCQ-Dfcb_VCrRawqN_8uWB1TocnRDZ_a3c1zazN-KOyP6c9-56PP3QOpczDj-Ui63y4r9HIeJw6IhJnmHj-K8wstuf2L
- japanshoreexcursions.com: https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGdNTLwdCbMqem8MTMhF-kIWzG8PYAkjGZ4K1_yV1QDtzlfKHAcUSpzDJDABBh7A8EEwMMWBcsR6FmBPCXpF6Bs5EoRVhpGUjqqMcwVbUIUuLFEWz4l069el1lJqoDNw2vyVmbhcEa8KYEnMf2vQg6Geict
- youtube.com: https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGCTqMhdS1Tk2KuHMxrTdGaEtprqgzl-ptoBgTzbvBta0EfZFoBs6tOnY065fLY1j8AVgvDCJwJVcIU1u3o44ntx8fnzQykQspLmDLAxgCnhIyn1b31YfV1G8glgJJ0-8uRMpMVSkA=
- alljapantours.com: https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGj7nx3T2H0YSswg01tZxzMfPGFoL91cAxs9Bx53Esa61aYSiUNCtonTCOp0NH-taxRBsLubfoJOc8Z2g-IjpFcrcyDKZz1sH3NNCpKAO8b

In [11]:
# Get video recommendation from YouTube using Gemini Search Grounding
import os
from IPython.display import Markdown, display

# 1. Read the list of cities/places from the generated itinerary
itinerary_file_path = "outputs/japan_itinerary.md"
if not os.path.exists(itinerary_file_path):
    itinerary_file_path = "notebooks/outputs/japan_itinerary.md"

try:
    with open(itinerary_file_path, "r") as f:
        itinerary_text = f.read()
except FileNotFoundError:
    itinerary_text = "Tokyo, Kyoto, Osaka, Hakone, Hiroshima" # Fallback

# 2. Read user preferences
prefs_file = "preferences.md"
if not os.path.exists(prefs_file):
    prefs_file = "notebooks/preferences.md"

try:
    with open(prefs_file, "r") as f:
        preferences_text = f.read()
except FileNotFoundError:
    preferences_text = "couple trip, young, mid luxury budget, love adventure, nature, history"

# 3. Create the prompt for YouTube grounding
youtube_prompt = f"""
I have the following Japan travel itinerary:
<itinerary>
{itinerary_text}
</itinerary>

My travel preferences are:
<preferences>
{preferences_text}
</preferences>

Please act as a travel video curator and use your search grounding to find the best YouTube videos for this trip.

1. **Top Travel Guides & Reviews:** Find popular travel experience videos covering the cities in my itinerary (prioritizing highly liked/viewed videos). Provide video titles, channels, and brief insights.
2. **Authentic Travelers & Hidden Gems:** Find videos from authentic travelers recommending hidden gems in or near these locations based on my preferences. List the specific hidden gems they mention.
3. **Know Before You Go:** Recommend popular YouTube videos about where to go, how to visit, and crucial things to know before going to Japan.

Format the response nicely in Markdown and include YouTube URLs where possible.
"""

# Configure Gemini to use Google Search Grounding to find YouTube videos
youtube_config = types.GenerateContentConfig(
    tools=[{"google_search": {}}],
    temperature=0.7,
)

print(f"Generating YouTube video recommendations with {MODEL_ID}...")
try:
    youtube_response = client.models.generate_content(
        model=MODEL_ID,
        contents=youtube_prompt,
        config=youtube_config
    )
    
    display(Markdown(youtube_response.text))
    
    # Save the output to a file
    output_dir = "outputs"
    if not os.path.exists(output_dir) and os.path.exists("notebooks/outputs"):
        output_dir = "notebooks/outputs"
    os.makedirs(output_dir, exist_ok=True)
    
    youtube_output_file = os.path.join(output_dir, "japan_youtube_recommendations.md")
    with open(youtube_output_file, "w") as f:
        f.write(youtube_response.text)
    print(f"\nSaved YouTube recommendations to {youtube_output_file}")
    
except Exception as e:
    print(f"An error occurred during generation: {e}")

Generating YouTube video recommendations with gemini-3.5-flash...


As a travel video curator, I have searched and handpicked the best YouTube videos to match your upcoming September trip to Japan. Since you are a young couple traveling on a **mid-luxury budget** with a passion for **adventure, nature, and history**, these videos focus on high-quality travel experiences, stunning scenic hikes, off-the-beaten-path cultural exploration, and essential practical tips.

---

### 1. Top Travel Guides & Reviews (Classic Hubs & Alpine Escapes)

These highly-rated travel guides capture the essence of your major hubs (Tokyo, Kyoto, Matsumoto, and Kamikochi), blending luxury comfort with outdoor thrills.

#### **[JAPAN TRAVEL | The Perfect Travel Itinerary For First-Timers](https://www.youtube.com/@worldwildhearts)**
*   **Creator:** World Wild Hearts
*   **Why Watch:** This video is perfect for active couples. Zac and Ine share a comprehensive itinerary that perfectly balances major cities with alpine nature. 
*   **Key Insights:** 
    *   **Matsumoto & Kamikochi:** They showcase how to do a seamless day trip to Kamikochi from Matsumoto, walking you through the breathtaking views of the Northern Alps and active volcano peaks.
    *   **Kyoto Trails:** They highlight how to escape the crowds in Kyoto by exploring the mountain trails and serene temples.

#### **[Japan Travel Guide: Best First-Timer Route + Itinerary](https://www.youtube.com/@alysmalls)**
*   **Creator:** Aly Smalls
*   **Why Watch:** Aly Smalls is fantastic for couples planning a mid-luxury trip. She focuses on boutique stays, high-end ryokans, and curated travel experiences.
*   **Key Insights:** 
    *   **Kyoto Ryokans:** She reviews premium traditional ryokan stays and onsen experiences.
    *   **Boutique Flow:** She shares how to structure your itinerary to minimize long, exhausting transit days so you can enjoy a romantic, stress-free trip.

---

### 2. Authentic Travelers & Hidden Gems

These videos are from authentic travelers who venture deep into the exact "hidden gems" on your itinerary, detailing the logistics of hiking, driving, and finding romantic, secluded spots.

#### **Yakushima Island: The Mossy Ghibli Forest**
*   **Video to watch:** **[Yakushima: Exploring Japan's Most Magical Island](https://www.youtube.com/@tatsukis)** (by Tatsuki)
*   **Specific Hidden Gems Mentioned:**
    *   **Shiratani Unsui Gorge:** The incredibly lush, moss-covered forest that served as the direct inspiration for Studio Ghibli’s *Princess Mononoke*.
    *   **Jōmon Sugi:** The oldest living cedar tree in Japan (estimated between 2,000 and 7,000 years old). Tatsuki shows you the grueling but deeply rewarding 8-to-12-hour hike to reach it.
    *   **Hirauchi Kaichu Onsen:** A wild, natural hot spring carved directly into the rocky coastline. It is entirely covered by the ocean and only reveals itself for a soak during low tide.

#### **The Iya Valley: Shikoku's Defeated Samurai Refuge**
*   **Video to watch:** **[Exploring Japan's Hidden Iya Valley](https://www.youtube.com/@maximos_travels)** (by Maximos Travels)
*   **Specific Hidden Gems Mentioned:**
    *   **Kazurabashi (Vine Bridge):** A thrilling and slightly terrifying crossing over a 1,000-year-old bridge woven from living mountain vines.
    *   **Nagoro (The Scarecrow Village):** A fascinating, surreal village where life-sized scarecrow dolls outnumber the actual residents.
    *   **Hotel Iya Onsen:** A secluded valley hotel where you must take a private cable car down a steep cliffside to reach the open-air hot spring baths next to a rushing river.

#### **The Nakasendo Trail: Walking with Samurais**
*   **Video to watch:** **[How to Hike the Nakasendo Trail (Tsumago to Magome)](https://www.youtube.com/@royandaimee)** (by Roy and Aimee)
*   **Specific Hidden Gems Mentioned:**
    *   **Magome-juku & Tsumago-juku:** Exceptionally preserved, car-free Edo-period post towns. They capture the beautiful waterwheels, cobblestone streets, and historic wooden buildings.
    *   **Odaki and Medaki (Male & Female Waterfalls):** A scenic stop along the 7.3km trail where legendary swordsman Miyamoto Musashi is said to have trained.
    *   **Traditional Tea Houses:** They show you where to take a break mid-hike to enjoy green tea and local sweets like *Gohei Mochi* (grilled rice cakes with sweet miso sauce).

---

### 3. September Festival Spotlight

#### **[Kishiwada Danjiri Festival: Japan's Most Dangerous Festival](https://www.youtube.com/@NipponTVNews24Japan)**
*   **Creator:** Nippon TV News 24 Japan
*   **Why Watch:** To prepare yourself for the wild, high-octane energy of the Danjiri Matsuri in mid-September. 
*   **What it shows:** Incredible footage of neighborhood teams sprinting through narrow streets hauling 4-ton wooden floats, executing high-speed, 90-degree turns (*Yari-mawashi*) while dancers leap on the roofs.

---

### 4. Know Before You Go: Crucial Travel Tips

These logistical videos are essential viewing before you depart, ensuring your transition into Japan’s unique culture and transport systems is flawless.

#### **[10 Essential Things to Know Before Travelling to Japan](https://www.youtube.com/@itssunnyinjapan)**
*   **Creator:** Sunny in Japan
*   **Crucial Tips for Your Itinerary:**
    *   **Luggage Forwarding (Takuhaibin):** Sunny explains how to ship your large suitcases from hotel to hotel. Since you are visiting remote areas like the Iya Valley and hiking the Nakasendo Trail, utilizing luggage forwarding is *mandatory* for a stress-free trip.
    *   **Digital IC Cards:** Learn how to instantly add a digital Suica card to your Apple Wallet before you land to tap through train gates seamlessly.
    *   **Onsen Etiquette:** Crucial rules for visiting luxury hot springs (bathing completely naked, washing thoroughly beforehand, and covering tattoos if required).

#### **[18 Things You MUST Know Before Visiting Tokyo](https://www.youtube.com/@itssunnyinjapan)**
*   **Creator:** Sunny in Japan
*   **Crucial Tips for Your Itinerary:**
    *   **Cash is King:** Even on a luxury budget, remote areas, local noodle shops, and temple stamp booths only accept physical cash.
    *   **Escalator Etiquette:** Stand on the left in Tokyo, but be prepared to stand on the right once you travel to Osaka for the Danjiri Festival!
    *   **Tax-Free Shopping:** Keep your physical passport on you at all times to claim instant tax refunds at high-end boutiques in Ginza or Shibuya.


Saved YouTube recommendations to outputs/japan_youtube_recommendations.md


In [12]:
import json

if response.candidates and response.candidates[0].grounding_metadata:
    metadata = response.candidates[0].grounding_metadata
    queries = getattr(metadata, 'web_search_queries', [])
    
    output = {
        "total_queries": len(queries),
        "queries": queries
    }
    
    print("Detailed structured output of web search queries:")
    print(json.dumps(output, indent=2))
else:
    print("No grounding metadata available.")


Detailed structured output of web search queries:
{
  "total_queries": 6,
  "queries": [
    "Japan hidden gems nature adventure history",
    "best places to visit Japan September adventure couple",
    "Kinchakuda Manjushage Park September",
    "Kishiwada Danjiri Matsuri September",
    "Sankara Hotel Yakushima",
    "Chiiori Iya Valley"
  ]
}


In [13]:
import os
import re
from IPython.display import Markdown, display

# Attempt to locate the previously generated YouTube recommendations
file_paths_to_try = [
    "outputs/japan_youtube_recommendations.md",
    "notebooks/outputs/japan_youtube_recommendations.md",
    "../notebooks/outputs/japan_youtube_recommendations.md"
]

youtube_content = ""
for path in file_paths_to_try:
    if os.path.exists(path):
        with open(path, "r") as f:
            youtube_content = f.read()
        break

if not youtube_content:
    print("Could not find japan_youtube_recommendations.md. Please ensure the YouTube recommendation cell has been run.")
else:
    # Extract the 'Know Before You Go' section
    # Match a heading containing "Know Before You Go" (ignoring case)
    # up to the next heading of the same or higher level, or end of string.
    match = re.search(r'(#{1,5}\s*.*?Know Before You Go.*?)(?=\n#{1,5}\s|\Z)', youtube_content, re.IGNORECASE | re.DOTALL)
    
    if not match:
        print("Could not find a 'Know Before You Go' section in the recommendations.")
    else:
        section_text = match.group(1)
        
        prompt = f"""
I have a section from a YouTube video recommendations document about 'Know Before You Go' for Japan.
Please read this text and extract all the actionable trip insights related to the trip that are mentioned in the videos. Format the insights nicely as a bulleted list in markdown.

Here is the section:
{section_text}
"""
        print(f"Extracting insights using Gemini...")
        try:
            insights_response = client.models.generate_content(
                model=MODEL_ID,
                contents=prompt
            )
            
            display(Markdown("### Extracted Trip Insights (Know Before You Go)\n\n" + insights_response.text))
            
            # Save the insights to a file
            output_dir = "outputs"
            if not os.path.exists(output_dir) and os.path.exists("notebooks/outputs"):
                output_dir = "notebooks/outputs"
            os.makedirs(output_dir, exist_ok=True)
            
            insights_file = os.path.join(output_dir, "japan_trip_insights.md")
            with open(insights_file, "w") as f:
                f.write("### Extracted Trip Insights (Know Before You Go)\n\n" + insights_response.text)
            print(f"\nSaved trip insights to {insights_file}")
            
        except Exception as e:
            print(f"Error during insights extraction: {e}")

Extracting insights using Gemini...


### Extracted Trip Insights (Know Before You Go)

Based on the provided video recommendations, here are the actionable trip insights extracted for your journey to Japan, categorized by destination and activity:

### **Itinerary & Route Planning**
*   **Pace Your Transit:** Structure your itinerary to minimize long, exhausting transit days to keep the trip romantic and stress-free.
*   **Combine Matsumoto and Kamikochi:** You can easily do a seamless day trip to Kamikochi from Matsumoto to experience the Northern Alps and active volcanic peaks.

### **Kyoto**
*   **Escape the Crowds:** Avoid heavy crowds in Kyoto by seeking out local mountain trails and serene, lesser-known temples.
*   **Indulge in Ryokans:** Book a premium traditional ryokan stay with an onsen (hot spring) experience while in Kyoto.

### **Yakushima Island**
*   **Explore the "Ghibli Forest":** Visit the incredibly lush, moss-covered **Shiratani Unsui Gorge**, which inspired the movie *Princess Mononoke*.
*   **Prepare for the Jōmon Sugi Hike:** If you want to see Japan's oldest living cedar tree, prepare and pack for a grueling **8-to-12-hour hike**.
*   **Check the Tides for Hirauchi Kaichu Onsen:** If you want to soak in this natural coastal hot spring, **you must time your visit during low tide**, as it is completely submerged by the ocean during high tide.

### **Iya Valley (Shikoku)**
*   **Cross the Vine Bridge:** Walk across **Kazurabashi**, a slightly terrifying bridge woven from living mountain vines suspended over the river.
*   **Visit the Scarecrow Village:** Take a surreal detour to **Nagoro**, a village where life-sized scarecrow dolls outnumber human residents.
*   **Stay at Hotel Iya Onsen:** Ride a private cable car down a steep cliffside to reach secluded, open-air hot spring baths right next to a rushing river.

### **The Nakasendo Trail**
*   **Hike the Post Towns:** Walk the historic **7.3km trail between Magome-juku and Tsumago-juku** to experience beautifully preserved, car-free Edo-period post towns.
*   **See the Waterfalls:** Make a scenic stop along the trail at **Odaki and Medaki** (Male & Female Waterfalls), where the legendary swordsman Miyamoto Musashi is said to have trained.
*   **Eat Local Mid-Hike:** Take a break at traditional tea houses along the trail to try **Gohei Mochi** (grilled rice cakes with sweet miso sauce) and fresh green tea.

### **September Festivals**
*   **Witness the Kishiwada Danjiri Festival:** If traveling in **mid-September**, head to Kishiwada (near Osaka) to witness one of Japan's most high-octane festivals, where teams sprint with 4-ton wooden floats through narrow streets. (Be prepared for massive crowds and high energy).


Saved trip insights to outputs/japan_trip_insights.md


## Grounding with Google Maps

If you want to explicitly use Google Maps for grounding (for instance, to get detailed location information, exact coordinates, or map links), you can define a tool for Google Maps (if supported by your specific API version) or use function calling with the Google Maps Places API.

The Gemini API is continually expanding its native tools. If your environment has native `google_maps` tool support enabled, it would look like this:

In [ ]:
import os

# Helper to read files safely regardless of where the notebook is run
def read_output_file(filename):
    paths = [f"outputs/{filename}", f"notebooks/outputs/{filename}", f"../notebooks/outputs/{filename}"]
    for p in paths:
        if os.path.exists(p):
            with open(p, "r") as f:
                return f.read()
    return f"[{filename} not found. Ensure previous cells were run.]"

itinerary_text = read_output_file("japan_itinerary.md")
youtube_text = read_output_file("japan_youtube_recommendations.md")

# Read user preferences
prefs_file = "preferences.md"
if not os.path.exists(prefs_file):
    prefs_file = "notebooks/preferences.md"

try:
    with open(prefs_file, "r") as f:
        preferences_text = f.read()
except FileNotFoundError:
    preferences_text = "trip, young, mid luxury budget, love adventure, nature, history"

maps_prompt = f"""
We have conducted initial research and compiled the following list of cities/places for a Japan trip:
<initial_research>
{itinerary_text}
</initial_research>

We also found the following YouTube recommendations, including popular spots and hidden gems:
<youtube_research>
{youtube_text}
</youtube_research>

Our travel preferences are:
<preferences>
{preferences_text}
</preferences>

Based on this previous research and our specific travel preferences, please use your Google Maps grounding to create an enriched, highly detailed, location-aware 14-day itinerary. 
Always first generate a section called 'Itinerary Overview and Route flow' to give the day and place breakdown of the entire planned itinerary.
Ensure the trip is realistic, logistically possible, and NOT rushed. Modify the transitions or pacing from the initial research to ensure the itinerary is truly feasible for a relaxing yet fulfilling trip, rather than rushing from place to place.
Do additional research on the specific hotspots, restaurants, and hidden gems mentioned above. Ensure the suggestions are physically near each other for practical routing, provide accurate transit/walking details, and confirm that the recommended cultural hotspots and adventure activities are top-rated on Maps and align closely with our preferences.

Please output the final, highly detailed itinerary formatted nicely in Markdown.
"""

# Configure the generation to use Google Maps Grounding specifically.
maps_config = types.GenerateContentConfig(
    tools=[ types.Tool(google_maps=types.GoogleMaps())],
    temperature=0.7,
)

print(f"Generating enriched itinerary with {MODEL_ID} and Maps grounding...")
try:
    maps_response = client.models.generate_content(
        model=MODEL_ID,
        contents=maps_prompt,
        config=maps_config
    )
    
    display(Markdown(maps_response.text))
    
    # Save the output to a file
    output_dir = "outputs"
    if not os.path.exists(output_dir) and os.path.exists("notebooks/outputs"):
        output_dir = "notebooks/outputs"
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, "japan_itinerary_maps.md")
    
    with open(output_file, "w") as f:
        f.write(maps_response.text)
    print(f"\nSaved Maps itinerary to {output_file}")
    
    # Optional: Display the search grounding sources if available
    if maps_response.candidates and maps_response.candidates[0].grounding_metadata:
        print("\n--- Maps Grounding Sources ---")
        metadata = maps_response.candidates[0].grounding_metadata
        if hasattr(metadata, 'grounding_chunks'):
            for chunk in metadata.grounding_chunks:
                if hasattr(chunk, 'web') and chunk.web:
                    print(f"- {chunk.web.title}: {chunk.web.uri}")
except Exception as e:
    print(f"An error occurred during generation: {e}")
    print("Note: If 'google_maps' is not natively supported in your current API version, it might default back to standard web search or raise a tool validation error.")

Generating enriched itinerary with gemini-3.5-flash and Maps grounding...


## Routing and Search Along the Route

Using the Google Maps grounding capability, we can also ask the model to plan specific routes, compare travel times, and find points of interest or restaurants along the way. This is particularly useful for planning day trips, scenic drives, or bullet train stops.

In [ ]:
import os
import re
from IPython.display import Markdown, display

# 1. Attempt to locate the previously generated maps itinerary
file_paths_to_try = [
    "outputs/japan_itinerary_maps.md",
    "notebooks/outputs/japan_itinerary_maps.md",
    "../notebooks/outputs/japan_itinerary_maps.md"
]

itinerary_maps_content = ""
for path in file_paths_to_try:
    if os.path.exists(path):
        with open(path, "r") as f:
            itinerary_maps_content = f.read()
        break

if not itinerary_maps_content:
    print("Could not find japan_itinerary_maps.md. Please ensure the Maps grounding cell has been run.")
    itinerary_overview = "### **Itinerary Overview & Route Flow**\n(Could not load dynamically - please run the previous cells.)"
else:
    # Extract the Itinerary Overview section using a highly flexible regex
    # It looks for headings containing words like "Overview", "Route", "Flow", or "Summary"
    # and captures everything until the next heading.
    match = re.search(r'(#+\s*[^#\n]*(?:Overview|Route|Flow|Summary|At a Glance)[^#\n]*\n.*?)(?=\n#+\s|\Z)', itinerary_maps_content, re.IGNORECASE | re.DOTALL)
    
    if match:
        itinerary_overview = match.group(1).strip()
        print("found the itinerary overview which was planned")
    else:
        itinerary_overview = "### **Itinerary Overview & Route Flow**\n(Could not find section in the generated itinerary.)"

In [ ]:
route_prompt = f"""
I have a 14-day trip planned with the following route overview:
{itinerary_overview}

Please use your Maps grounding to provide a comprehensive transportation guide for moving between these locations and compare it with the golden route(Tokyo -> Hakone -> Kyoto -> Hiroshima/Miyajima -> Osaka).

For each leg of the journey:
1. Provide all possible options of travel (including bullet train/shinkansen, local public transport, and driving), along with estimated travel times.
2. For the driving option, find specific travel stops along the way. This should include recommendations for where to fill up on gas, highly-rated restaurants to grab a bite, and nearby scenic attractions or viewpoints where we can stop the car and enjoy the experience.
3. Format the response nicely in Markdown with clear headings for each route segment.
4. Highlight places that are logistically impossible or if the trip feels too rushed
"""

print(f"Generating comprehensive route information with {MODEL_ID} and Maps grounding...")
try:
    # We reuse maps_config which has tools=[{"google_maps": {}}] configured
    route_response = client.models.generate_content(
        model=MODEL_ID,
        contents=route_prompt,
        config=maps_config
    )
    
    display(Markdown(route_response.text))
    
    # Save the output to a file
    output_dir = "outputs"
    if not os.path.exists(output_dir) and os.path.exists("notebooks/outputs"):
        output_dir = "notebooks/outputs"
    os.makedirs(output_dir, exist_ok=True)
    
    output_file = os.path.join(output_dir, "japan_comprehensive_routing.md")
    with open(output_file, "w") as f:
        f.write(route_response.text)
    print(f"\nSaved Comprehensive Routing to {output_file}")
    
    # Optional: Display the search grounding sources if available
    if route_response.candidates and route_response.candidates[0].grounding_metadata:
        print("\n--- Maps Grounding Sources ---")
        metadata = route_response.candidates[0].grounding_metadata
        if hasattr(metadata, 'grounding_chunks'):
            for chunk in metadata.grounding_chunks:
                if hasattr(chunk, 'web') and chunk.web:
                    print(f"- {chunk.web.title}: {chunk.web.uri}")
                else:
                    print("no grounding info on")
except Exception as e:
    print(f"An error occurred during generation: {e}")

Generating comprehensive route information with gemini-3.5-flash and Maps grounding...


This custom 14-day itinerary is a phenomenal, adventure-focused alternative to the standard tourist path. Below is a comparison of this route against the traditional "Golden Route," followed by a comprehensive leg-by-leg transportation guide and a logistical reality check.

---

### 🗺️ Route Comparison: Your Route vs. The Golden Route

| Feature | The Golden Route (Tokyo ➔ Hakone ➔ Kyoto ➔ Hiroshima ➔ Osaka) | Your Route (Tokyo ➔ Gunma ➔ Nagano ➔ Toyama ➔ Osaka) |
| :--- | :--- | :--- |
| **Primary Vibe** | Urban, historical temples, neon lights, and classic shrines. | High-adrenaline adventure, alpine hiking, sacred forests, and coastal glass art. |
| **Crowd Levels** | **Very High.** Prone to heavy overtourism, especially in Kyoto and Hakone. | **Low to Moderate.** Deeply authentic, with mostly domestic tourists and nature lovers. |
| **Transit Ease** | **Extremely Easy.** Dominated by the ultra-fast, direct Tokaido Shinkansen. | **Complex.** Involves crossing the Japanese Alps, requiring winding mountain roads, local buses, or multi-transfer train routes. |
| **Best Travel Mode**| Shinkansen (JR Pass is highly convenient here). | **Car Rental** is highly recommended for the alpine legs (Days 2–11), switching to trains in Osaka. |

---

### ⚠️ Logistical Feasibility & "Too Rushed" Warnings

While highly rewarding, this itinerary is packed. Keep these crucial logistics in mind:

1. **The Gunma-to-Nagano Mountain Barrier (Day 5):** Minakami (Gunma) and Togakushi (Nagano) are separated by a massive mountain range. By public transport, this requires multiple transfers and takes half a day. If driving, you must navigate winding toll roads. 
2. **Kamikochi Private Car Ban (Day 8):** **You cannot drive a private vehicle into Kamikochi.** If you rent a car, you must park it at the Sawando Parking Area and take a shuttle bus or taxi.
3. **Kishiwada Danjiri Matsuri Timing (Day 13):** The actual adrenaline-fueled *Danjiri Matsuri* only takes place on specific weekends in **September and October**. If your trip is at any other time of the year, the town will be quiet, but you can still visit the **Kishiwada Danjiri Hall** to see the massive floats. During the actual festival, driving to Kishiwada is virtually impossible due to gridlock and road closures—you must take the train.

---

### 🚗 Leg-by-Leg Transportation Guide

---

#### 📍 Leg 1: Tokyo to Minakami (Gunma)
*The transition from the neon metropolis to the rugged valleys of Tone River.*

*   **Option A: Shinkansen & Bus (Fastest)**
    *   **Route:** Joetsu Shinkansen from Tokyo Station to Jomo-Kogen Station (~75 mins), then a local Kan-Etsu Kotsu Bus to Minakami Station (~25 mins).
    *   **Total Time:** ~1 hour 40 mins.
*   **Option B: Local Trains (Cheapest)**
    *   **Route:** JR Takasaki Line from Ueno to Takasaki Station, then transfer to the JR Joetsu Line to Minakami Station.
    *   **Total Time:** ~2.5 to 3 hours.
*   **Option C: Driving**
    *   **Route:** Take the Kan-Etsu Expressway (E17) north directly from Tokyo to the Minakami IC.
    *   **Total Time:** ~2 hours 15 mins (depending on Tokyo exit traffic).
    *   **🚗 Driving Stops:**
        *   *Scenic Attraction:* **Lake Haruna / Mt. Haruna** (A slight detour off the Shibukawa-Ikaho IC, famous as the setting for the anime *Initial D*).
        *   *Highly-Rated Restaurant:* **Oshimizu** or **Tamaruya** on the famous *Mizusawa Udon Street* near Shibukawa. Mizusawa Udon is famous for being thick, cold, and incredibly chewy.
        *   *Gas Station:* **ENEOS Shibukawa Ikaho Inter** (Conveniently located right off the highway exit before entering the mountains).

---

#### 📍 Leg 2: Minakami (Gunma) to Togakushi (Nagano)
*A scenic journey crossing the mountain borders into the mystical cedar forests of Nagano.*

*   **Option A: Train & Bus**
    *   **Route:** JR Joetsu Line from Minakami to Takasaki (~1 hour). Transfer to the Hokuriku Shinkansen from Takasaki to Nagano Station (~45 mins). From Nagano Station, take the Alpico Bus (Route 70) to Togakushi (~1 hour).
    *   **Total Time:** ~3 to 3.5 hours (requires tight scheduling).
*   **Option B: Driving (Highly Recommended for this leg)**
    *   **Route:** Kan-Etsu Expressway (E17) south to Fujioka JCT, then Joshin-Etsu Expressway (E18) to Nagano IC, and local Route 37/406 up the mountain to Togakushi.
    *   **Total Time:** ~2.5 to 3 hours.
    *   **🚗 Driving Stops:**
        *   *Scenic Attraction:* **Obuse Town**. A historic, beautifully preserved merchant town famous for its chestnut orchards, traditional wood-and-plaster buildings, and the Hokusai Museum.
        *   *Highly-Rated Restaurant:* **Uzuraya** (located right next to Togakushi Shrine Chusha). It is legendary for Togakushi Soba, served in traditional *bochi-mori* (five small woven bundles). *Note: Queues can be up to 2 hours; arrive early to write your name on the list.*
        *   *Gas Station:* **ENEOS Nagano Inter** (Fill up in Nagano City before making the steep climb up to the Togakushi plateau).

---

#### 📍 Leg 3: Togakushi (Nagano) to Matsumoto (Nagano)
*Descending from the sacred highlands to the historic castle town of Matsumoto.*

*   **Option A: Bus & Train**
    *   **Route:** Alpico Bus from Togakushi back to Nagano Station (~1 hour). Transfer to the JR Shinano Limited Express to Matsumoto Station (~50 mins).
    *   **Total Time:** ~2 hours.
*   **Option B: Driving**
    *   **Route:** Local mountain roads down to Nagano City, then the Nagano Expressway (E19) south from Nagano IC to Matsumoto IC.
    *   **Total Time:** ~1.5 to 2 hours.
    *   **🚗 Driving Stops:**
        *   *Scenic Attraction:* **Daio Wasabi Farm** in Azumino. One of Japan's largest wasabi farms, featuring pristine, gravel-filtered alpine streams and picturesque wooden waterwheels.
        *   *Highly-Rated Restaurant:* **Nomugi** in Matsumoto. A highly-rated, rustic shop famous for serving authentic, handmade 100% buckwheat *juwari soba*.
        *   *Gas Station:* **ENEOS Azumino Inter** (Right off the expressway exit, before entering Matsumoto city center).

---

#### 📍 Leg 4: Matsumoto to Kamikochi (Day Trip)
*A journey into the heart of the Chubu-Sangaku National Park.*

*   **Option A: Train & Bus (Easiest)**
    *   **Route:** Matsumoto Dentetsu Kamikochi Line train from Matsumoto Station to Shin-Shimashima Station (~30 mins). Transfer directly to the Alpico Bus to Kamikochi Bus Terminal (~65 mins).
    *   **Total Time:** ~1 hour 45 mins (one way).
*   **Option B: Driving (With Restrictions)**
    *   **🔴 CRITICAL WARNING:** **Private cars are strictly banned in Kamikochi.** 
    *   **Route:** Drive from Matsumoto along Route 158 to the **Sawando Parking Area** (~1 hour). Park your car (¥700/day) and board the official Kamikochi Shuttle Bus or take a shared taxi to the Kamikochi Bus Terminal (~30 mins).
    *   **Total Time:** ~1 hour 30 mins.
    *   **🚗 Driving Stops:**
        *   *Scenic Attraction:* **Roadside Station Fuketsu-no-sato** (道の駅 風穴の里) along Route 158. Stop here to see the natural cold-air caves (historical natural refrigerators) and buy local Nagano apples.
        *   *Highly-Rated Restaurant:* **Gosenjaku Hotel Kamikochi** (located right next to the iconic Kappabashi Bridge). Their restaurant is famous for its luxurious Shinshu beef curry and fresh-baked apple pie.
        *   *Gas Station:* **ENEOS Shin-Shimashima** (This is your absolute last chance to fill up before heading into the long, dark tunnels of Route 158).

---

#### 📍 Leg 5: Matsumoto to Toyama City
*Crossing the roof of Japan to the Sea of Japan coast.*

*   **Option A: Direct Highway Bus (Most Convenient)**
    *   **Route:** Direct highway bus operated by Alpico/Toyama Chiho Railway from Matsumoto Bus Terminal to Toyama Station.
    *   **Total Time:** ~3 hours 15 mins.
*   **Option B: The Scenic "Alpine Route" (Epic but expensive)**
    *   **Route:** Tateyama Kurobe Alpine Route (Matsumoto ➔ Shinano-Omachi ➔ Ogisawa ➔ Tateyama ➔ Toyama). This involves cable cars, trolley buses, and ropeways crossing right through the mountains.
    *   **Total Time:** 7 to 8 hours (A full-day transit; you must forward your luggage separately).
*   **Option C: Driving**
    *   **Route:** Drive west via Route 158 through the Abo Tunnel, pass through Takayama, and then head north on Route 41 to Toyama.
    *   **Total Time:** ~3 hours.
    *   **🚗 Driving Stops:**
        *   *Scenic Attraction:* **Takayama Old Town (Sanmachi Suji)**. A beautifully preserved merchant district with dark-wood Edo-period buildings and historic sake breweries.
        *   *Highly-Rated Restaurant:* **Maruaki** in Takayama. Famous for serving premium, melt-in-your-mouth Hida Beef (Wagyu) which you can grill yourself at the table.
        *   *Gas Station:* **ENEOS Takayama Inter** (Fill up before taking Route 41 north through the mountain valleys toward Toyama).

---

#### 📍 Leg 6: Toyama to Osaka
*Transitioning from the tranquil northern coast to the neon-lit food capital of Kansai.*

*   **Option A: Shinkansen & Limited Express (Fastest)**
    *   **Route:** Hokuriku Shinkansen from Toyama to Tsuruga (~1 hour). Transfer directly to the JR Thunderbird Limited Express from Tsuruga to Osaka (~1 hour 25 mins).
    *   **Total Time:** ~2 hours 45 mins.
*   **Option B: Driving**
    *   **Route:** Take the Hokuriku Expressway (E8) south along the coast to Maibara JCT, then transition to the Meishin Expressway (E1) to Osaka.
    *   **Total Time:** ~4 to 4.5 hours.
    *   **🚗 Driving Stops:**
        *   *Scenic Attraction:* **Shirahige Shrine** on Lake Biwa. A stunning Shinto shrine featuring a "floating" red torii gate in the waters of Japan's largest lake.
        *   *Highly-Rated Restaurant:* **Sennari-tei Kyogoku-ten** in the nearby historic town of Hikone. Famous for serving Omi Beef (one of Japan's top three Wagyu brands) as beef sushi or sukiyaki.
        *   *Gas Station:* **ENEOS Amagozen Service Area** (A great highway rest stop with ocean views on the Hokuriku Expressway).

---

#### 📍 Leg 7: Osaka to Kishiwada (Day Trip)
*A quick excursion south to the home of the wild Danjiri festival.*

*   **Option A: Nankai Railway (Highly Recommended)**
    *   **Route:** Nankai Main Line (Express or Limited Express) from Namba Station in Osaka directly to Kishiwada Station.
    *   **Total Time:** ~25 to 30 mins.
*   **Option B: Driving**
    *   **Route:** Take the Hanshin Expressway Route 4 (Wangan Route) south along the bay.
    *   **Total Time:** ~45 mins.
    *   **🔴 WARNING:** **If traveling during the actual Danjiri Matsuri weekend, do not drive.** The entire town is pedestrianized, traffic is deadlocked, and there is absolutely zero public parking. 
    *   **🚗 Driving Stops (Non-Festival Days Only):**
        *   *Scenic Attraction:* **Kishiwada Castle**. A beautiful castle featuring the *Hachijin-no-niwa* (Eight Triads Garden), a unique modern rock garden designed by Mirei Shigeori.
        *   *Highly-Rated Restaurant:* **Ganko Kishiwada Goko-so**. A stunning, historic family mansion with a magnificent stroll-garden serving traditional Japanese kaiseki and sushi.
        *   *Gas Station:* **ENEOS Kishiwada-Minami IC** (Conveniently located near the highway exit).

---

#### 📍 Leg 8: Osaka to Tokyo (Day 14 Return Option)
*Heading back to the capital to catch your flight home.*

*   **Option A: Tokaido Shinkansen (Fastest)**
    *   **Route:** Shinkansen *Nozomi* from Shin-Osaka Station to Tokyo Station.
    *   **Total Time:** ~2.5 hours.
*   **Option B: Overnight Highway Bus (Cheapest)**
    *   **Route:** Direct highway bus from Osaka Umeda/Namba to Shinjuku/Tokyo Station.
    *   **Total Time:** ~8 to 9 hours (leaves around 10:30 PM, arrives at 6:30 AM).
*   **Option C: Driving**
    *   **Route:** Take the Shin-Tomei Expressway (E1A) east.
    *   **Total Time:** ~5.5 to 6 hours.
    *   **🚗 Driving Stops:**
        *   *Scenic Attraction:* **NEOPASA Surugawan-Numazu Service Area**. An upscale highway rest stop designed to look like a Mediterranean villa, offering breathtaking panoramic views of Mt. Fuji and Suruga Bay.
        *   *Highly-Rated Restaurant:* **Sawayaka** (Locations are scattered near expressway exits in Shizuoka prefecture). This is Shizuoka's cult-classic, legendary charcoal-grilled hamburger steak restaurant. *Be prepared for long wait times.*
        *   *Gas Station:* **ENEOS Hamamatsu Service Area** (A massive, clean service area on the Shin-Tomei Expressway).


Saved Comprehensive Routing to outputs/japan_comprehensive_routing.md

--- Maps Grounding Sources ---
no grounding info on
no grounding info on
no grounding info on
no grounding info on
no grounding info on
no grounding info on
no grounding info on
no grounding info on
no grounding info on


In [16]:
route_response.candidates[0].grounding_metadata.grounding_chunks

[GroundingChunk(
   maps=GroundingChunkMaps(
     place_id='places/ChIJX7Vx8JRrHmARb5DI7ugZ_KA',
     title='Jomokogen Station',
     uri='https://maps.google.com/?cid=11600175228428783727'
   )
 ),
 GroundingChunk(
   maps=GroundingChunkMaps(
     place_id='places/ChIJRakCoyRgHmARg4yVHhBfp3U',
     title='Kan-etsu Kotsu',
     uri='https://maps.google.com/?cid=8477849346385480835'
   )
 ),
 GroundingChunk(
   maps=GroundingChunkMaps(
     place_id='places/ChIJz9ShsinqHmAR03WjF1ZOztQ',
     title='Kamisato Service Area for Tokyo',
     uri='https://maps.google.com/?cid=15334279912913860051'
   )
 ),
 GroundingChunk(
   maps=GroundingChunkMaps(
     place_id='places/ChIJefMprL0UHmAR2K3-OvGLna8',
     title='Suwakyo Gorge',
     uri='https://maps.google.com/?cid=12654424396174110168'
   )
 ),
 GroundingChunk(
   maps=GroundingChunkMaps(
     place_id='places/ChIJ69wo1YsOHWARD73yPYb8Kto',
     title='Matsumoto Station',
     uri='https://maps.google.com/?cid=15720655102785273103'
   )
 ),

import os
import re
import json
import requests
from IPython.display import Markdown, display

# 1. Load config for Maps API Key
MAPS_API_KEY = "YOUR_MAPS_API_KEY"

for config_path in ['config.json', '../config.json']:
    if os.path.exists(config_path):
        with open(config_path, 'r') as f:
            config_data = json.load(f)
            MAPS_API_KEY = config_data.get('apiKeys', {}).get('maps', MAPS_API_KEY)
        break

# 2. Attempt to locate the previously generated maps itinerary
file_paths_to_try = [
    "outputs/japan_itinerary_maps.md",
    "notebooks/outputs/japan_itinerary_maps.md",
    "../notebooks/outputs/japan_itinerary_maps.md"
]

itinerary_content = ""
output_dir = "outputs"

for path in file_paths_to_try:
    if os.path.exists(path):
        with open(path, "r") as f:
            itinerary_content = f.read()
        output_dir = os.path.dirname(path)
        break

if not itinerary_content:
    print("Could not find japan_itinerary_maps.md. Please ensure the Maps grounding cell has been run.")
else:
    # 3. Extract places using a heuristic: bold text like **TRUNK(HOTEL) YOYOGI PARK**
    raw_places = re.findall(r'\*\*([^\*]+)\*\*', itinerary_content)
    
    ignore_list = ["Morning", "Afternoon", "Evening", "Lunch", "Dinner", "Arrival", "Check-in", "September", "Route Overview", "Total Travel Time", "Route", "Gas Station", "Highly-Rated Restaurant", "Scenic Attraction / Viewpoint"]
    places_to_search = []
    for p in raw_places:
        if any(ign in p for ign in ignore_list) or "Day " in p or p.startswith("Days ") or "AM" in p or "PM" in p or ":" in p:
            continue
        if p.strip() and p.strip() not in places_to_search:
            places_to_search.append(p.strip())
            
    # For demonstration, limit to first 30 places to avoid long execution times and API rate limits
    places_to_search = places_to_search[:30]
            
    print(f"Extracted potential places. Processing first {len(places_to_search)} places for deep review summaries...")
    
    results_md = "# Deep Dive: Reviews & Photos\n\n"
    
    # Create temporary directory for images (not in outputs)
    temp_img_dir = "temp_images"
    os.makedirs(temp_img_dir, exist_ok=True)
    
    # 4. Use Google Maps New Places API
    search_url = "https://places.googleapis.com/v1/places:searchText"
    
    def extract_text(val):
        if not val: return None
        if isinstance(val, dict):
            if "text" in val: return val["text"]
            if "overview" in val and isinstance(val["overview"], dict): return val["overview"].get("text", str(val))
            return json.dumps(val)
        return str(val)
    
    for place_name in places_to_search:
        # Search for place
        search_payload = {
            "textQuery": f"{place_name} Japan"
        }
        search_headers = {
            "X-Goog-Api-Key": MAPS_API_KEY,
            "X-Goog-FieldMask": "places.id,places.displayName,places.formattedAddress,places.rating",
            "Content-Type": "application/json"
        }
        
        search_res = requests.post(search_url, json=search_payload, headers=search_headers).json()
        
        if search_res.get("error"):
            print(f"API Error for {place_name}: {search_res['error'].get('message', search_res['error'])}")
            results_md += f"### {place_name} (Search Failed due to API Error)\n"
            results_md += f"- Error: {search_res['error'].get('message', 'Unknown Error')}\n\n---\n"
            continue
            
        if search_res.get("places"):
            place = search_res["places"][0]
            place_id = place.get("id")
            name_obj = place.get("displayName", {})
            name = name_obj.get("text", place_name) if isinstance(name_obj, dict) else place_name
            address = place.get("formattedAddress", "No address")
            rating = place.get("rating", "N/A")
            
            # Get extensive place details including new summaries
            details_url = f"https://places.googleapis.com/v1/places/{place_id}"
            details_headers = {
                "X-Goog-Api-Key": MAPS_API_KEY,
                "X-Goog-FieldMask": "id,displayName,reviews,photos,websiteUri,generativeSummary,areaSummary,reviewSummary,evChargeOptions,accessibilityOptions,parkingOptions"
            }
            det_res = requests.get(details_url, headers=details_headers).json()
            
            image_md = ""
            positive_md = "No positive reviews available.\n"
            negative_md = "No negative reviews available.\n"
            
            if "id" in det_res:
                website = det_res.get("websiteUri", "No website")
                reviews = det_res.get("reviews", [])
                photos = det_res.get("photos", [])
                
                # New Summaries
                gen_summary = extract_text(det_res.get("generativeSummary"))
                area_summary = extract_text(det_res.get("areaSummary"))
                rev_summary = extract_text(det_res.get("reviewSummary"))
                
                ev_options = det_res.get("evChargeOptions")
                ev_md = json.dumps(ev_options) if ev_options else None
                
                parking = det_res.get("parkingOptions")
                access = det_res.get("accessibilityOptions")
                amenities = []
                if parking: amenities.append(f"Parking: {json.dumps(parking)}")
                if access: amenities.append(f"Accessibility: {json.dumps(access)}")
                amenities_md = " | ".join(amenities) if amenities else None
                
                # Extract top 1 image and save it locally using New Places API photos URL
                if photos:
                    photo_name = photos[0].get("name") # Format: places/{placeId}/photos/{photoReference}
                    if photo_name:
                        photo_url = f"https://places.googleapis.com/v1/{photo_name}/media?maxHeightPx=400&maxWidthPx=400&key={MAPS_API_KEY}"
                        
                        # Download image to temp folder
                        img_response = requests.get(photo_url)
                        if img_response.status_code == 200:
                            img_filename = f"{place_id}.jpg"
                            img_path = os.path.join(temp_img_dir, img_filename)
                            with open(img_path, "wb") as img_file:
                                img_file.write(img_response.content)
                            
                            # Use relative path one level up since this file will be saved in outputs/
                            image_md = f"![{name}](../{img_path})\n\n"
                        else:
                            # Fallback to URL if download fails
                            image_md = f"![{name}]({photo_url})\n\n"
                
                if reviews:
                    # Sort reviews by rating
                    sorted_reviews = sorted(reviews, key=lambda x: x.get("rating", 0))
                    
                    positive_reviews = [r for r in sorted_reviews if r.get("rating", 0) >= 4]
                    negative_reviews = [r for r in sorted_reviews if r.get("rating", 0) < 3]
                    
                    # Top 10 positive (highest to lowest)
                    top_positive = sorted(positive_reviews, key=lambda x: x.get("rating", 0), reverse=True)[:10]
                    # Top 10 negative (lowest to highest)
                    top_negative = sorted(negative_reviews, key=lambda x: x.get("rating", 0))[:10]
                    
                    if top_positive:
                        positive_md = ""
                        for r in top_positive:
                            r_rating = r.get("rating")
                            r_text_obj = r.get("text", {})
                            r_text = r_text_obj.get("text", "") if isinstance(r_text_obj, dict) else ""
                            r_text = r_text.replace("\n", " ")
                            positive_md += f"- {r_rating} Stars: {r_text}\n"
                            
                    if top_negative:
                        negative_md = ""
                        for r in top_negative:
                            r_rating = r.get("rating")
                            r_text_obj = r.get("text", {})
                            r_text = r_text_obj.get("text", "") if isinstance(r_text_obj, dict) else ""
                            r_text = r_text.replace("\n", " ")
                            negative_md += f"- {r_rating} Stars: {r_text}\n"
            
            results_md += f"### {name}\n"
            results_md += image_md
            results_md += f"- **Place ID:** `{place_id}`\n"
            results_md += f"- **Address:** {address}\n"
            results_md += f"- **Overall Rating:** {rating} ⭐\n"
            
            if gen_summary:
                results_md += f"- **Generative Summary:** {gen_summary}\n"
            if area_summary:
                results_md += f"- **Neighborhood/Area Summary:** {area_summary}\n"
            if rev_summary:
                results_md += f"- **Review Summary:** {rev_summary}\n"
            if amenities_md:
                results_md += f"- **Amenities Summary:** {amenities_md}\n"
            if ev_md:
                results_md += f"- **EV Charge Options:** {ev_md}\n"
                
            results_md += f"#### Top 10 Positive Reviews:\n{positive_md}\n"
            results_md += f"#### Top 10 Negative/Lowest Reviews:\n{negative_md}\n\n"
            results_md += "---\n"
        else:
            results_md += f"### {place_name} (Search Failed)\n"
            results_md += f"- No Place ID found.\n\n---\n"
            
    display(Markdown(results_md))
    
    # Save the output
    os.makedirs(output_dir, exist_ok=True)
    detailed_output_file = os.path.join(output_dir, "japan_place_reviews.md")
    with open(detailed_output_file, "w") as f:
        f.write(results_md)
    print(f"\nSaved advanced reviews to {detailed_output_file}")

In [43]:
import os
import re
import json
import requests
from IPython.display import Markdown, display

# 1. Load config for Maps API Key
MAPS_API_KEY = "YOUR_MAPS_API_KEY"

for config_path in ['config.json', '../config.json']:
    if os.path.exists(config_path):
        with open(config_path, 'r') as f:
            config_data = json.load(f)
            MAPS_API_KEY = config_data.get('apiKeys', {}).get('maps', MAPS_API_KEY)
        break

# 2. Attempt to locate the previously generated maps itinerary
file_paths_to_try = [
    "outputs/japan_itinerary_maps.md",
    "notebooks/outputs/japan_itinerary_maps.md",
    "../notebooks/outputs/japan_itinerary_maps.md"
]

itinerary_content = ""
output_dir = "outputs"

for path in file_paths_to_try:
    if os.path.exists(path):
        with open(path, "r") as f:
            itinerary_content = f.read()
        output_dir = os.path.dirname(path)
        break

if not itinerary_content:
    print("Could not find japan_itinerary_maps.md. Please ensure the Maps grounding cell has been run.")
else:
    # 3. Extract places using a heuristic: bold text like **TRUNK(HOTEL) YOYOGI PARK**
    raw_places = re.findall(r'\*\*([^\*]+)\*\*', itinerary_content)
    
    ignore_list = ["Morning", "Afternoon", "Evening", "Lunch", "Dinner", "Arrival", "Check-in", "September", "Route Overview", "Total Travel Time", "Route", "Gas Station", "Highly-Rated Restaurant", "Scenic Attraction / Viewpoint"]
    places_to_search = []
    for p in raw_places:
        if any(ign in p for ign in ignore_list) or "Day " in p or p.startswith("Days ") or "AM" in p or "PM" in p or ":" in p:
            continue
        if p.strip() and p.strip() not in places_to_search:
            places_to_search.append(p.strip())
            
    # For demonstration, limit to first 30 places to avoid long execution times and API rate limits
    places_to_search = places_to_search[:30]
            
    print(f"Extracted potential places. Processing first {len(places_to_search)} places for deep review summaries...")
    
    results_md = "# Deep Dive: Reviews & Photos\n\n"
    
    # Create temporary directory for images (not in outputs)
    temp_img_dir = "temp_images"
    os.makedirs(temp_img_dir, exist_ok=True)
    
    # 4. Use Google Maps New Places API
    search_url = "https://places.googleapis.com/v1/places:searchText"
    
    def extract_text(val):
        if not val: return None
        if isinstance(val, dict):
            if "text" in val: return val["text"]
            if "overview" in val and isinstance(val["overview"], dict): return val["overview"].get("text", str(val))
            return json.dumps(val)
        return str(val)
    
    for place_name in places_to_search:
        # Search for place
        search_payload = {
            "textQuery": f"{place_name} Japan"
        }
        search_headers = {
            "X-Goog-Api-Key": MAPS_API_KEY,
            "X-Goog-FieldMask": "places.id,places.displayName,places.formattedAddress,places.rating",
            "Content-Type": "application/json"
        }
        
        search_res = requests.post(search_url, json=search_payload, headers=search_headers).json()
        
        if search_res.get("error"):
            print(f"API Error for {place_name}: {search_res['error'].get('message', search_res['error'])}")
            results_md += f"### {place_name} (Search Failed due to API Error)\n"
            results_md += f"- Error: {search_res['error'].get('message', 'Unknown Error')}\n\n---\n"
            continue
            
        if search_res.get("places"):
            place = search_res["places"][0]
            place_id = place.get("id")
            name_obj = place.get("displayName", {})
            name = name_obj.get("text", place_name) if isinstance(name_obj, dict) else place_name
            address = place.get("formattedAddress", "No address")
            rating = place.get("rating", "N/A")
            
            # Get extensive place details including new summaries
            details_url = f"https://places.googleapis.com/v1/places/{place_id}"
            details_headers = {
                "X-Goog-Api-Key": MAPS_API_KEY,
                "X-Goog-FieldMask": "id,displayName,reviews,photos,websiteUri,generativeSummary,areaSummary,reviewSummary,evChargeOptions,accessibilityOptions,parkingOptions"
            }
            det_res = requests.get(details_url, headers=details_headers).json()
            
            image_md = ""
            positive_md = "No positive reviews available.\n"
            negative_md = "No negative reviews available.\n"
            
            if "id" in det_res:
                website = det_res.get("websiteUri", "No website")
                reviews = det_res.get("reviews", [])
                photos = det_res.get("photos", [])
                
                # New Summaries
                gen_summary = extract_text(det_res.get("generativeSummary"))
                area_summary = extract_text(det_res.get("areaSummary"))
                rev_summary = extract_text(det_res.get("reviewSummary"))
                
                ev_options = det_res.get("evChargeOptions")
                ev_md = json.dumps(ev_options) if ev_options else None
                
                parking = det_res.get("parkingOptions")
                access = det_res.get("accessibilityOptions")
                amenities = []
                if parking: amenities.append(f"Parking: {json.dumps(parking)}")
                if access: amenities.append(f"Accessibility: {json.dumps(access)}")
                amenities_md = " | ".join(amenities) if amenities else None
                
                # Extract top 1 image and save it locally using New Places API photos URL
                if photos:
                    photo_name = photos[0].get("name") # Format: places/{placeId}/photos/{photoReference}
                    if photo_name:
                        photo_url = f"https://places.googleapis.com/v1/{photo_name}/media?maxHeightPx=400&maxWidthPx=400&key={MAPS_API_KEY}"
                        
                        # Download image to temp folder
                        img_response = requests.get(photo_url)
                        if img_response.status_code == 200:
                            img_filename = f"{place_id}.jpg"
                            img_path = os.path.join(temp_img_dir, img_filename)
                            with open(img_path, "wb") as img_file:
                                img_file.write(img_response.content)
                            
                            image_md = f"![{name}]({img_path})\n\n"
                        else:
                            # Fallback to URL if download fails
                            image_md = f"![{name}]({photo_url})\n\n"
                
                if reviews:
                    # Sort reviews by rating
                    sorted_reviews = sorted(reviews, key=lambda x: x.get("rating", 0))
                    
                    positive_reviews = [r for r in sorted_reviews if r.get("rating", 0) >= 4]
                    negative_reviews = [r for r in sorted_reviews if r.get("rating", 0) < 3]
                    
                    # Top 10 positive (highest to lowest)
                    top_positive = sorted(positive_reviews, key=lambda x: x.get("rating", 0), reverse=True)[:10]
                    # Top 10 negative (lowest to highest)
                    top_negative = sorted(negative_reviews, key=lambda x: x.get("rating", 0))[:10]
                    
                    if top_positive:
                        positive_md = ""
                        for r in top_positive:
                            r_rating = r.get("rating")
                            r_text_obj = r.get("text", {})
                            r_text = r_text_obj.get("text", "") if isinstance(r_text_obj, dict) else ""
                            r_text = r_text.replace("\n", " ")
                            positive_md += f"- {r_rating} Stars: {r_text}\n"
                            
                    if top_negative:
                        negative_md = ""
                        for r in top_negative:
                            r_rating = r.get("rating")
                            r_text_obj = r.get("text", {})
                            r_text = r_text_obj.get("text", "") if isinstance(r_text_obj, dict) else ""
                            r_text = r_text.replace("\n", " ")
                            negative_md += f"- {r_rating} Stars: {r_text}\n"
            
            results_md += f"### {name}\n"
            results_md += image_md
            results_md += f"- **Place ID:** `{place_id}`\n"
            results_md += f"- **Address:** {address}\n"
            results_md += f"- **Overall Rating:** {rating} ⭐\n"
            
            if gen_summary:
                results_md += f"- **Generative Summary:** {gen_summary}\n"
            if area_summary:
                results_md += f"- **Neighborhood/Area Summary:** {area_summary}\n"
            if rev_summary:
                results_md += f"- **Review Summary:** {rev_summary}\n"
            if amenities_md:
                results_md += f"- **Amenities Summary:** {amenities_md}\n"
            if ev_md:
                results_md += f"- **EV Charge Options:** {ev_md}\n"
                
            results_md += f"#### Top 10 Positive Reviews:\n{positive_md}\n"
            results_md += f"#### Top 10 Negative/Lowest Reviews:\n{negative_md}\n\n"
            results_md += "---\n"
        else:
            results_md += f"### {place_name} (Search Failed)\n"
            results_md += f"- No Place ID found.\n\n---\n"
            
    display(Markdown(results_md))
    
    # Save the output
    os.makedirs(output_dir, exist_ok=True)
    detailed_output_file = os.path.join(output_dir, "japan_place_reviews.md")
    with open(detailed_output_file, "w") as f:
        f.write(results_md)
    print(f"\nSaved advanced reviews to {detailed_output_file}")

Extracted potential places. Processing first 30 places for deep review summaries...


# Deep Dive: Reviews & Photos

### Kishiwada Danjiri (Festival Float) Hall
![Kishiwada Danjiri (Festival Float) Hall](temp_images/ChIJ8UkEOwzGAGARo4MASnxj16c.jpg)

- **Place ID:** `ChIJ8UkEOwzGAGARo4MASnxj16c`
- **Address:** 11-23 Honmachi, Kishiwada, Osaka 596-0074, Japan
- **Overall Rating:** 4.1 ⭐
- **Review Summary:** {'text': 'Visitors say this museum offers a close-up view of elaborate danjiri floats and their intricate carvings, along with engaging theater presentations and hands-on taiko drumming experiences. They also highlight the helpful staff and the convenient combined ticket option with Kishiwada Castle.\n\nSome reviews mention the information can be unclear.', 'languageCode': 'en-US'}
- **Amenities Summary:** Accessibility: {"wheelchairAccessibleParking": true, "wheelchairAccessibleEntrance": true, "wheelchairAccessibleRestroom": true}
#### Top 10 Positive Reviews:
- 5 Stars: This is one of the most famous festivals in Japan. It was so cool and fun actually. The most exciting part was where the large danjiri are pulled at high speed around corners. There is carpenter (Daikugata) dancing on top of the gloat while it is moving. There's a vibrant and exhilarating atmosphere filled with rythmic drumming, flute music, and chants of the danjiri pullers
- 5 Stars: It was really hard looking for info in English on the Kishiwada Danjiri Festival Map or parade route online when we went in 2024 so I'm uploading it here!
- 5 Stars: Nice museum where you can have a close see to Daijiri (vehicles of matsuri)
- 5 Stars: Great environment, inetersting knowledge, fantastic experience !!
- 5 Stars: Nice cultural experience!

#### Top 10 Negative/Lowest Reviews:
No negative reviews available.


---
### Nui. Hostel & Bar Lounge
![Nui. Hostel & Bar Lounge](temp_images/ChIJ4U-9KsiOGGARARhaBLZLqS0.jpg)

- **Place ID:** `ChIJ4U-9KsiOGGARARhaBLZLqS0`
- **Address:** 2-chōme-14-13 Kuramae, Taito City, Tokyo 111-0051, Japan
- **Overall Rating:** 4.5 ⭐
- **Review Summary:** {'text': "People say this hostel offers clean, comfortable rooms with privacy curtains and spacious, well-maintained shared bathrooms. They also highlight the delicious food and coffee available at the lively downstairs cafe and bar, which attracts both locals and travelers. Guests mention the friendly, helpful staff and the hostel's convenient location near public transport and local attractions.", 'languageCode': 'en-US'}
- **Amenities Summary:** Accessibility: {"wheelchairAccessibleParking": false}
#### Top 10 Positive Reviews:
- 5 Stars: Nui. is the perfect hostel for people who want a little bit of everything. The rooms had a perfect level of people between them, the beds were comfortable and everything was extremely clean. I loved that the downstairs café/bar was filled with locals and tourists alike making an incredible atmosphere when you arrive or when you want to have a few drinks after a long day of sightseeing making it easy to socialise with others. I ended up meeting so many people during my stay who I'm lucky enough to call my friends.  If you want to get away from the hustle and bustle and take in a great view of Tokyo Skytree and Sumida River, head to the top floor where there is a balcony.  The hostel is located right next to Asakusa where Sensō-ji temple is located making it a great spot for sightseeing, restaurants and bars. If you would like to head out for the day to another district, the subway is a 5 minute walk away making it easy to get around stress-free.  Big shoutout to Maki-san, Miku-san, Sora-san and all the bar staff for making my stay here as amazing as possible.  I genuinely cannot express how amazing of a stay I had here but one thing's for sure. I'll definitely be back in the future!
- 5 Stars: I didn’t stay at the hostel, but a friend of mine did and she enjoyed her experience. This was a nice place to meet up in the morning before heading out on our Tokyo adventures. They have a great soy latte and the atmosphere lends itself to making friends. While waiting one morning I made friends with someone else stopping over for a coffee and we all went to hangout together afterwards.
- 5 Stars: Really solid for a classic hostel. The 8 person mixed gender dorm room is clean, each bed is surrounded by privacy curtains which really help with the noise/light. The AC works great! The staff is all so friendly and kind. It’s on a quiet side street and it was really nice to come back to a less busy part of town each night for some calmness.
- 5 Stars: I highly recommend this place. It was great location imo (quiet neighborhood but ten min walk to Asakusa or twenty min train to Ginza or Shibuya). Room was so clean and they have curtains for the dorms. Bathroom also very clean. The one critique I have is that it was not as social as I was hoping for and people kinda ignored me when I tried to talk to them with like one exception. I’ve never experienced that before and I am super respectful so I promise it’s not just me. I’m not sure if that’s usually how it’s like at this place but regardless, I still highly recommend as everything that was in Nui’s control was beyond expectations.

#### Top 10 Negative/Lowest Reviews:
- 2 Stars: I ordered the rigatoni because they were the only dish on the menu that caught my attention and, most importantly, they did not have the spicy symbol. Since I have stomach issues, I chose them thinking it was a safe option. Unfortunately, the pasta turned out to be spicy. When I pointed this out to the waiter, he denied that it was spicy instead of checking or being helpful. I’m giving two stars because the pasta was well cooked and the ingredients were of good quality. However, the bartender’s attitude was not appropriate toward a customer who was simply pointing out a mistake on the menu, especially regarding something as important as spiciness.


---
### UNPLAN Shinjuku
![UNPLAN Shinjuku](temp_images/ChIJCd85CYONGGAROzjK0EgWo9E.jpg)

- **Place ID:** `ChIJCd85CYONGGAROzjK0EgWo9E`
- **Address:** 5-chōme-3-15 Shinjuku, Shinjuku City, Tokyo 160-0022, Japan
- **Overall Rating:** 4.2 ⭐
- **Amenities Summary:** Accessibility: {"wheelchairAccessibleParking": false}
#### Top 10 Positive Reviews:
- 5 Stars: I really recommend this hotel ! We stayed in a 3 person room, with one double bed and 2 futons.  The room was quite big for Tokyo. We could fit our 3 big luggages. The breakfast was perfect, which good bread. There was salty options : eggs, sausages, salad.  The commune bathroom are super clean !!  The location was perfect. There is a 7-Eleven in from of the hotel.  The - : - we had a private bedroom, but if you take a room in a dormitory, i can imagine it could be difficult to chill in the lobby in the night with all the people chatting very loudly. - the room was noisy, you can hear the street and people could be noisy in the corridor at night. But I don’t think it’s relevant since every hotel we did in Japan was noisy. It was perfectly fine with earplugs. - the curtains doesn’t hide the sun so bring a sleeping mask - the hairdryer doesn’t dry anything
- 5 Stars: I stayed at Unplan Shinjuku and honestly, it's a place I can't wait to return to.  The common areas are fantastic. There's always a great mix of people from all sorts of countries and backgrounds, so striking up a conversation comes naturally — even if you're a bit shy, you'll fit right in. They also host events regularly, which meant my longer stay never got boring; in fact, I looked forward to every day. I made genuine friends here, and we even went out for drinks together after the stay.  What really stood out, though, was the quality of the beds. Even as a dormitory, each bunk is built solidly and gives you a real sense of private space. The bed was perfectly comfortable — no complaints at all. Compared to the many hostels I've stayed at around the world, this place is on a completely different level.  On top of that, breakfast is included for free — something genuinely rare for hostels in Japan, and a lovely way to start the morning.  Ease of meeting people, quality of the facilities, value for money — it nails all three. Whether you're looking to connect with fellow travelers or just want a comfortable, well-made place to rest, I can recommend Unplan Shinjuku without hesitation.
- 5 Stars: I stayed in a few places on my trip, and this was absolutely my favorite hostel. It felt clean and safe, and I really liked the bed space. I was able to put my packing cubes and stuff in there and felt like I had plenty of room, in a cozy way. Shower always had hot water, breakfast was actually very convenient and nice, and the vibes overall were just great. Location was perfect too, the area was nice and just a short walk to so many cool bars. And, I wanted to be near a big station (Shinjuku), but also a smaller one so I didn’t have to traverse Shinjuku every time, and that was perfect. I stayed a week in the female dorm, and I’d do that again in a heartbeat.
- 5 Stars: Stayed in the mixed dorm in March 2026. Hostel is decently close to most of the attractions in Shinjuku.  Beds and rooms are clean. Bathrooms are spotless. There is a safe in every bunk with a removable key. The free breakfast was nice and had variety. Each floor requires access via pin code for added security. Definitely would recommend for other travellers with a budget in the low to mid range.

#### Top 10 Negative/Lowest Reviews:
No negative reviews available.


---
### Takasaki Station
![Takasaki Station](temp_images/ChIJh3ZzfZuSHmARSljV8hrRWhE.jpg)

- **Place ID:** `ChIJh3ZzfZuSHmARSljV8hrRWhE`
- **Address:** Yashimacho, Takasaki, Gunma 370-0849, Japan
- **Overall Rating:** 3.9 ⭐
- **Review Summary:** {'text': 'People say this station is a convenient transportation hub with numerous train lines, including Shinkansen, and a wide variety of stores and restaurants inside. Visitors also highlight the clean and well-maintained facilities, as well as the helpful staff.\n\nOther reviews mention the staff can be impolite.', 'languageCode': 'en-US'}
- **Amenities Summary:** Parking: {"paidParkingLot": true} | Accessibility: {"wheelchairAccessibleParking": true, "wheelchairAccessibleEntrance": true, "wheelchairAccessibleRestroom": true}
#### Top 10 Positive Reviews:
- 5 Stars: Great Joetsu shinkansen and local JR rail station, well connected  and situated to the wonderful city of Takasaki with lots of nearby stores, restaurants, and lodgings.  As always, JR Rail keeps the station clean and efficient, and, also as always, the trains do run on time.  I was especially impressed with the large courtyard area outside the front of the station, which offered places to hang out and chat with friends.  There is a large department store adjacent to the courtyard, as well as an APA hotel.  The connection of the station to the city is really well done overall.  Clean and wonderful.
- 5 Stars: It’s a really convenient station in Takasaki for both bus and trains! You can go to Tokyo and Narita directly from here! We always use this place for our trips! Around the station, there are bunch of stores and restaurants to enjoy and do shopping! Keep up the great work! Cheers 🥂
- 5 Stars: This train station links up to hotels and mall which definitely great. There's a lot of shop too inside the station building. It's not hard for foreigner like me to seek for directions with the great clear sign boards around. I bought my ticket at the counter, the lady was very helpful despite the language barrier we facing.
- 5 Stars: A bustling cradle of travel where city rhythm meets gentle countryside promise. Trains slide out like quiet thoughts heading west, leaving Takasaki’s lively hum behind. The station feels like a lantern in broad daylight — bright, welcoming, and ready to send you along a slower, greener path toward silk towns and mountain breaths. 🌆🚉🍃  Tourist Tip: Great place to gather supplies before rural stretches. Clear signage for transfers; expect crowds at peak times. 🧭 The Takasaki Line rests here, its long journey fulfilled. Final stop. Visit Takasaki’s Daruma Temple and Mount Kannon. Transfer to Joetsu or Shinetsu Lines for mountain travel. The city offers both art and serenity.  Joshin Line: From Takasaki’s lively hum I drift into quieter air, the little Jōshin cars carrying me past fields brushed with sun. Yamana’s shrine wind greets me, Tomioka whispers its silk-spun history, and hills gather close as the rails wind toward Shimonita’s cool valley. Each stop feels like a gentle hand guiding me deeper into Gunma’s warm countryside—where slow travel becomes a kind of blessing. 🌾🍃🚉
- 4 Stars: Quite a large train station but it’s conveniently linked with buses/stops to bring one to the surrounding areas. There are also souvenir shops n food outlets(mall) here so it’s easy to catch a meal before your bus ride. Most bus Drivers can speak a little English but are very accommodating n helpful. It’s not difficult to catch the right bus, just inquire politely n always confirm your destination when boarding the bus with the driver. The train station is itself comfortable and nice to browse around for 30 minutes or have a meal break or just to pick up some souvenirs! The Daruma Bento is a good buy because after eating one can retain the red plastic container (which is a Daruma) for a souvenir keepsake.

#### Top 10 Negative/Lowest Reviews:
No negative reviews available.


---
### Joshin-etsu Expressway
![Joshin-etsu Expressway](temp_images/ChIJ8TI1rUbkHWARpgKmEgIfxls.jpg)

- **Place ID:** `ChIJ8TI1rUbkHWARpgKmEgIfxls`
- **Address:** Joshin-etsu Expy, Japan
- **Overall Rating:** N/A ⭐
#### Top 10 Positive Reviews:
No positive reviews available.

#### Top 10 Negative/Lowest Reviews:
No negative reviews available.


---
### Minakami Station
![Minakami Station](temp_images/ChIJwySJbX0UHmARsdqeJ9Jo4Mk.jpg)

- **Place ID:** `ChIJwySJbX0UHmARsdqeJ9Jo4Mk`
- **Address:** Kanosawa, Minakami, Tone District, Gunma 379-1611, Japan
- **Overall Rating:** 3.8 ⭐
- **Amenities Summary:** Accessibility: {"wheelchairAccessibleParking": true, "wheelchairAccessibleEntrance": true, "wheelchairAccessibleRestroom": true}
#### Top 10 Positive Reviews:
- 5 Stars: Japanese people here are friendly and not afraid of foreigners. Went for a day trip to see a snow and and twice on different situation, one at the station and one the the park nearby approached us and offer to take the photo of us ( with wife). Been living in Japan for already 10 years and this is unusual and I like it.
- 5 Stars: I went there few days before it was snowing i enjoy but there is not so many peoples and almost it seems ghost town lol
- 5 Stars: This is a small train station and it has everything you would expect. The bathrooms are large clean and relatively recently redone. There are vending machines with hot and cold drinks. There's a covered outdoor sitting area with a taxi stand and, surprisingly, a taxi.
- 4 Stars: A station that is easy to reach from Tokyo and also convenient to go to many attractions around Minakami. You can get on the bus from here to get to the Tanigawadake Ropeway Station (and enjoy powder snow at the top during winter) or get the combination bus ticket for Takaragawa Onsen at the Tourist Information center right beside the platform gate. There are many other options, do check with the tourist info lady~ The downside is that other than a few souvenir shops in front of the station, all the other cafe and restaurants are either closed down or closed really early (they were closed around 2pm+ during winter) so there is no place to pass time other than the station's waiting room...
- 4 Stars: Neat small station with comfortable waiting room, ticket machines and manned counter with helpful staff who provided change. Coin lockers are available both inside the waiting room and outside, along with toilets. Nowhere to buy takeaway snacks in the immediate vicinity. There are also apparently some heritage train events here

#### Top 10 Negative/Lowest Reviews:
No negative reviews available.


---
### Tone River
![Tone River](temp_images/ChIJDdVWrnlpHmARmXuwYMrkb9Q.jpg)

- **Place ID:** `ChIJDdVWrnlpHmARmXuwYMrkb9Q`
- **Address:** Tone River, Japan
- **Overall Rating:** 3.8 ⭐
#### Top 10 Positive Reviews:
- 5 Stars: Best sunset viewing spots.
- 5 Stars: From inside the car.  This is the Tone River, which I've only ever seen on maps.  Amazing!
- 5 Stars: This time we're back with the "To" part of the Jomo Karuta series. Tone is the greatest river in Bando. The Tone River is the greatest river in Bando, and as the eldest son of all Japan's rivers, he is affectionately known as Bando Taro, meaning the most sacred river in Japan. It is followed by Chikushi Jiro (Chikushi River) and Shikoku Saburo (Yoshino River). Ever since I was a child, I remember playing in the river until dark every day. I've been reminded once again of the greatness of the Tone River. Thank you.
- 5 Stars: We took a drive along the Tone River with our family this past weekend, enjoying cycling and a picnic along the way. The wide river and the expansive view of the sky stretching into the distance were even more breathtaking than in pictures. The breeze on top of the embankment was refreshing, and the seasonally changing flowers and plants along the banks, along with the seagulls and herons gliding across the water, made for a picturesque scene. There were well-maintained parks and benches scattered throughout, allowing us to relax while the children played. In spring, there are cherry blossom trees; in summer, fireworks and water play; in autumn, golden rice fields and sunsets; and in winter, crisp air and starry skies. No matter how many times you visit, the scenery is different, and you never get tired of walking the same places.  There were also many people fishing, canoeing, and road cycling, making it a perfect field for outdoor enthusiasts. There are some sections with fast currents, so those with children should be careful of life jackets and the "no entry" signs along the river. Toilets and parking vary depending on the area, so it's best to check a map app beforehand. There are also sections without nearby convenience stores, so bring plenty of drinks, sun protection, and insect repellent. The rules for barbecuing along the riverbank vary by section, so following the signs will ensure a smooth and enjoyable experience.  Despite its easy access from the city center, the area boasts a vast sky and a tranquil atmosphere. In the evening, the silhouette of the bridges spanning the Tone River against the reddish-orange glow was breathtakingly beautiful, making you want to capture it on camera. While close to nature, the area is well-maintained, making it perfect for beginners. It's a place I'd love to visit again in different seasons and would recommend to family and friends.  Early morning walks are also highly recommended. The moment the sun shines on the misty river surface is incredibly quiet, with only the sounds of birdsong and water. The cycling path has a relatively clean surface, and the straight sections with good visibility are easy even for beginners. However, there are some steep downhill sections along the embankment, so be careful not to speed. History buffs will enjoy observing the river's flood control measures, such as the levees, sluice gates, and weirs. Seasonal event information is posted on the local bulletin board, giving a warm and welcoming feeling to the community.   Things to note are that on windy days, dust can easily be blown around, and there are sections with little shade. Bringing a hat, sunglasses, and a picnic blanket is recommended. If you're bringing a pet, don't forget a trash bag and water. If everyone follows the rules, everyone can enjoy themselves. Drones and bonfires are prohibited in many areas, so check the rules beforehand. Overall, it was a river space where nature and human activity blended together nicely, a place I'd want to visit again and again. Next time, I plan to go in the spring when the rapeseed blossoms are in bloom, bringing a packed lunch.
- 4 Stars: This is the Tone River as seen from Shibukawa City during a trip to Gunma. 😽 Rivers are one of the joys of traveling, but the great thing about the Tone River is that because it spans prefectures, you can see its many different faces. ✨

#### Top 10 Negative/Lowest Reviews:
No negative reviews available.


---
### 天神ロッジ (Tenjin Lodge)
![天神ロッジ (Tenjin Lodge)](temp_images/ChIJ6Xn9VMkWHmARMlaXO_izH2I.jpg)

- **Place ID:** `ChIJ6Xn9VMkWHmARMlaXO_izH2I`
- **Address:** 220-4 Yubiso, Minakami, Tone District, Gunma 379-1728, Japan
- **Overall Rating:** 4.6 ⭐
#### Top 10 Positive Reviews:
- 5 Stars: We stayed at Tenjin Lodge for 1 night during our visit to the Doai and Tanigawadake area. The lodge is located about 10–15 minutes walk from Doai Station, around another 10 minutes to the Tanigawadake Information Center, and about 15 minutes walk to the Tanigawadake Ropeway station. The location is very convenient for hikers, skiers and nature lovers exploring this beautiful mountain region.  Compared to many traditional Japanese guesthouses nearby, Tenjin Lodge has a very different atmosphere. It is run by an Australian owner and international staff, giving the place a cozy mountain lodge vibe that feels both relaxed and welcoming. After several nights sleeping on futons during our Japan trip, it was such a nice surprise to finally sleep on a comfortable king-size bed 😄.  The common area is spacious and warm, with large dining tables, comfortable sofas, bookshelves filled with interesting books, a piano, and even a hammock. The lodge sits next to the Yubiso River, so throughout the day you can hear the relaxing sound of flowing water outside.  Downstairs there is a Japanese-style bathing area with a large hot bath, perfect after a day of walking or hiking. Meals were simple but delicious, and there is also a small bar corner where guests can enjoy sake, beer or wine while relaxing at night. One of the highlights of the stay was Lucky, the friendly lodge dog who quickly became everyone’s furry companion during the stay.  The owner, Kieren, was also extremely kind and even helped fetch us to Yubiso Station, which we truly appreciated.  If you are planning to hike, ski, visit Doai Station or simply enjoy the peaceful mountain atmosphere around Tanigawadake, Tenjin Lodge is a warm and comfortable place to stay.
- 5 Stars: The lodge is a vibe! Stay here if you want to meet other snow enthusiasts. Kieren and his staff work hard to provide guests with an experience, so you'll have a great time if you come with a good attitude. The dinner is amazing- big hearty, healthy meals that fill you up after a big day of riding. The rooms are traditional and functional and worked well for us.  We did 3 days of guiding with Patrick and he was super fun and knowledgeable to ride with. He made sure we got plenty of fresh powder, and was professional throughout.  Thank you Tenjin Lodge for a memorable stay!
- 5 Stars: Nice cozy lodge, particularly like the bar living room, very chilled out. Sauna nice and hot! Western beds comfy and warm shower. They also do guiding around Mt T, hiked up to the peak, they really know the area and are very safety conscious.
- 5 Stars: Came there as a last resort if i couldnt make it back to toyko after climbing mt tanigawa. Just wanted some place to rest my head, but got a full experience with sauna, cold plunge, onsen and Kirin even took us to the local bar. There is wildlife there and the stars at night are great. Kirin is a top bloke and would put this one my to do list if your in the area

#### Top 10 Negative/Lowest Reviews:
No negative reviews available.


---
### Canyons キャニオンズ
![Canyons キャニオンズ](temp_images/ChIJUeEp5kYUHmAR0AolF0JkqdU.jpg)

- **Place ID:** `ChIJUeEp5kYUHmAR0AolF0JkqdU`
- **Address:** 45 Yubiso, Minakami, Tone District, Gunma 379-1728, Japan
- **Overall Rating:** 4.8 ⭐
- **Amenities Summary:** Parking: {"freeParkingLot": true, "paidParkingLot": false, "freeStreetParking": false, "paidStreetParking": false, "freeGarageParking": false, "paidGarageParking": false}
#### Top 10 Positive Reviews:
- 5 Stars: Me and my wife an am amazing experience. The staff were friendly and knowledgeable. We felt safe the entire time while also having a thrilling experience. If you find yourself craving a little bit of of the outdoors and a bit of adrenaline then this is a great way to spend the day. Big shout out to Tom who made our adventure special and fun.
- 5 Stars: The most important things for rafting is safety, and you can see the whole team a very professional, we are rafting with two boats, and there are one guy to escort us along the whole journey to ensure our safety. Our guy Sam is a mixed, and native in both English and Japanese, so communication is not a problem, the whole journey goes through an extremely scenic gorge, the best rafting experience ever
- 5 Stars: We loved the canyoning and packrafting with Canyons! Packrafting down the small rapids was fun and surprisingly manageable. Our guides Daiki, Ashok, and Adam were great! Thanks for accommodating our last minute booking for the packrafting, and we also appreciated the hotel pickup and dropoff.
- 5 Stars: We had a full day of fun of river Canyon Fox plus and River rafting. The staff was great and helpful. Would definitely recommend to do this activities and really experiencing something you never did before Will certainly do it again!!
- 5 Stars: We had an amazing experience with Canyons. This is a well-run company that provides great service and great experiences. We took the Whitewater Rafting/Canyoning combo experience in Minakami and loved it. Our guides were awesome and it was the highlight of our Japan trip. Highly recommend Canyons. They take care of everything.

#### Top 10 Negative/Lowest Reviews:
No negative reviews available.


---
### Canyons キャニオンズ
![Canyons キャニオンズ](temp_images/ChIJUeEp5kYUHmAR0AolF0JkqdU.jpg)

- **Place ID:** `ChIJUeEp5kYUHmAR0AolF0JkqdU`
- **Address:** 45 Yubiso, Minakami, Tone District, Gunma 379-1728, Japan
- **Overall Rating:** 4.8 ⭐
- **Amenities Summary:** Parking: {"freeParkingLot": true, "paidParkingLot": false, "freeStreetParking": false, "paidStreetParking": false, "freeGarageParking": false, "paidGarageParking": false}
#### Top 10 Positive Reviews:
- 5 Stars: Me and my wife an am amazing experience. The staff were friendly and knowledgeable. We felt safe the entire time while also having a thrilling experience. If you find yourself craving a little bit of of the outdoors and a bit of adrenaline then this is a great way to spend the day. Big shout out to Tom who made our adventure special and fun.
- 5 Stars: The most important things for rafting is safety, and you can see the whole team a very professional, we are rafting with two boats, and there are one guy to escort us along the whole journey to ensure our safety. Our guy Sam is a mixed, and native in both English and Japanese, so communication is not a problem, the whole journey goes through an extremely scenic gorge, the best rafting experience ever
- 5 Stars: We loved the canyoning and packrafting with Canyons! Packrafting down the small rapids was fun and surprisingly manageable. Our guides Daiki, Ashok, and Adam were great! Thanks for accommodating our last minute booking for the packrafting, and we also appreciated the hotel pickup and dropoff.
- 5 Stars: We had a full day of fun of river Canyon Fox plus and River rafting. The staff was great and helpful. Would definitely recommend to do this activities and really experiencing something you never did before Will certainly do it again!!
- 5 Stars: We had an amazing experience with Canyons. This is a well-run company that provides great service and great experiences. We took the Whitewater Rafting/Canyoning combo experience in Minakami and loved it. Our guides were awesome and it was the highlight of our Japan trip. Highly recommend Canyons. They take care of everything.

#### Top 10 Negative/Lowest Reviews:
No negative reviews available.


---
### Yubiso Station
![Yubiso Station](temp_images/ChIJP0RlWUcUHmAR1V63l8iLqFw.jpg)

- **Place ID:** `ChIJP0RlWUcUHmAR1V63l8iLqFw`
- **Address:** Yubiso, Minakami, Tone District, Gunma 379-1728, Japan
- **Overall Rating:** 4.3 ⭐
- **Review Summary:** {'text': 'Visitors say this train station features unique underground platforms for southbound trains and offers views of trains descending a loop line from the northbound platform. They also highlight the ease of access to the underground platform, noting it has fewer stairs than other similar stations. Guests mention the station is surrounded by peaceful nature and is the closest stop for Yubiso Onsen.', 'languageCode': 'en-US'}
- **Amenities Summary:** Accessibility: {"wheelchairAccessibleParking": false, "wheelchairAccessibleEntrance": false}
#### Top 10 Positive Reviews:
No positive reviews available.

#### Top 10 Negative/Lowest Reviews:
No negative reviews available.


---
### Kubota
![Kubota](temp_images/ChIJWYha5QcPHWARSQj_ccf-uiU.jpg)

- **Place ID:** `ChIJWYha5QcPHWARSQj_ccf-uiU`
- **Address:** 10-17 Ichiba, Matsumoto, Nagano 399-0004, Japan
- **Overall Rating:** 3.9 ⭐
- **Amenities Summary:** Parking: {"freeParkingLot": true} | Accessibility: {"wheelchairAccessibleParking": false, "wheelchairAccessibleEntrance": false, "wheelchairAccessibleSeating": false}
#### Top 10 Positive Reviews:
- 5 Stars: It's great that they offer complimentary homemade Kyuchan and large portions of tempura set meals at reasonable prices. Because the area is home to many workers, the kakaeshi and tempura bowls are quite strong in flavor. If you want to eat a lot of soba, you should go to a restaurant away from the city center.
- 5 Stars: Kubota's Tempura Zaru (Matsumoto City, Nagano Prefecture)  Amidst the glossy soba noodles, the crispy, oily flavor moistens the mouth.  Tonkotsu Shotaro  When I go to a soba restaurant, I usually order tempura. Biting into it between bites of the bland soba leaves a hint of oil on the rim. Slurping the soba with the oil adds a richer flavor, making it even more delicious.  At Kubota in Matsumoto City, Shinshu. A colleague told me it was delicious, so I immediately went there. A restaurant beloved by locals. Looking at the menu, I found that in addition to soba, they also offer udon, rice bowls, a la carte dishes, and more, all packed with ideas to keep you entertained.  Since it was my first visit, I started with the classic tempura Zaru. I took a bite of the glossy, smooth, truly "hand-made" soba noodles. Delicious. The aroma of soba gently wafts from my mouth to my nose, bringing a smile to my face. After two or three bites, I bite into the eggplant tempura. Crunchy.  Then I return to the soba. I slurp, slurping as if to wash away the fine oil that has clung to my lips. When the broth and oil combine in my mouth, the mild flavor of the soba is magnified twice as much. Excellent.  Next time I'd like to try the rice bowl as well. Thank you for the meal.  #HandmadeKubota #HandmadeSoba #Soba #Shinshu #MatsumotoCity #Handmade #TonkotsuShotaro #GourmetTanka #Gourmet #BusinessTrip #BusinessTripGourmet #FoodieConnects #Yummy
- 4 Stars: Located in a corner of the market, this well-established restaurant has been in business for around 30 years, but its immaculately clean interior and hand-made soba noodles are a local favorite, renowned for their "great value for money." The vegetable tempura soba I had this time was as generous as rumored. The tempura, made with plenty of seasonal vegetables, had a crispy batter and a wide variety of options, making it very satisfying to eat.   The main dish, soba noodles, are unevenly thick, with the flavor and firm texture you'd expect from hand-made noodles, making it a delicious dish with a pleasant texture. The friendly staff are also very welcoming, but perhaps due to the careful preparation, the kitchen seems to be run by a single person. The restaurant is so busy that the parking lot fills up during weekday lunchtime, so it's best to visit with plenty of time to relax and enjoy a delicious bowl of noodles.

#### Top 10 Negative/Lowest Reviews:
- 2 Stars: Mixed reviews… Seeing is believing Please see and taste for yourself… I had a sudden craving for Japanese soba noodles and decided to go to a restaurant I'd never been to before. I generally prefer to go spontaneously without doing any research beforehand, so I deliberately avoided reading reviews. I finally got to enjoy the soba I'd been looking forward to, but while noodle tastes are subjective, the noodles were so watery that I wondered if they'd been properly drained, and the dipping sauce was diluted. A bit disappointing… The tempura was said to take some time to prepare, but it upset my stomach so much I couldn't eat dinner. There was also a rice bowl set, which this time was meatballs, which might be good for those who prioritize volume…


---
### Tanigawadake Joch by Hoshino Resorts
![Tanigawadake Joch by Hoshino Resorts](temp_images/ChIJJSz0C80WHmARlxi8fMSxOnw.jpg)

- **Place ID:** `ChIJJSz0C80WHmARlxi8fMSxOnw`
- **Address:** Japan, 〒379-1728 Gunma, Tone District, Minakami, Yubiso, 吹山国有林
- **Overall Rating:** 4.2 ⭐
- **Review Summary:** {'text': 'People say this mountain resort offers stunning panoramic views of the Tanigawa mountain range and features well-maintained hiking trails suitable for all levels. They also highlight the delicious local specialty, the "pan gratin," and the clean, well-maintained restrooms. Others also like the friendly and helpful staff, and the convenient indoor parking.', 'languageCode': 'en-US'}
- **Amenities Summary:** Parking: {"paidGarageParking": true} | Accessibility: {"wheelchairAccessibleParking": true, "wheelchairAccessibleEntrance": true}
#### Top 10 Positive Reviews:
- 5 Stars: I came  here on 10/21/23 to take photo of mountain Koyo. Because is not clear blue sky day I was lucky to see rainbow. The experience of sitting at cable car alone with colourful autumn leaves around was absolutely amazing !! Then rainbow did a magic touch  to the whole landscape  !! You can buy set ticket to go up to the top cost 3500yen including cable car and a lift . I was advice not to take lift since it is raining . At the 1319m top I saw many hikers , they told me 4 hours to hike to the top. My ticket cost 2000yen round trip . I should come back in summer to do the hike . If you like colourful autumn leaves 🍁 it is highly recommended. Only 2 hours from tokyo station, easy to do a day trip .
- 5 Stars: A grueling but highly rewarding winter hike with spectacular views.  Winter hike not advised for beginners.  Cramp ons, ice axe required, ski poles highly recommended.
- 4 Stars: We visited the Tenjindaira Observatory, transformed into a silver world by the heavy snowfall the night before. We took the ropeway and lift up, and the hike to the top took about 10 minutes. We enjoyed a 360-degree view. Perhaps because it was right after the snowfall, the air was incredibly clear. We could see Mt. Tanigawa right in front of us, and even distant peaks like the Yatsugatake and Mt. Fuji.
- 4 Stars: Beautiful ski resort and mountain. A little expensive. A little dated. Great ropeway and view. Many lifts and runs. Not crowded and fun. Good restaurant with nice workers.
- 4 Stars: Great experience. On a clear day you can see many distant mountains. Wish I had come better prepared for a little hike but there was so much snow.

#### Top 10 Negative/Lowest Reviews:
No negative reviews available.


---
### Mount Tanigawa
![Mount Tanigawa](temp_images/ChIJmzCINf0QHmARoy3beRJ09SI.jpg)

- **Place ID:** `ChIJmzCINf0QHmARoy3beRJ09SI`
- **Address:** Mount Tanigawa, Yubiso, Minakami, Tone District, Gunma 379-1728, Japan
- **Overall Rating:** 4.7 ⭐
#### Top 10 Positive Reviews:
- 5 Stars: Excellent place to check out. Quite an easy and short drive from Echigo Yuzawa station along the Joetsu Shinkansen line. Went in June and it was still rather chilly (about 10-12C in the day).
- 5 Stars: I climbed through Nishiguro ridge. the wind was strong and I can't see the trail path because of the clouds. I almost died. never forget that experience
- 5 Stars: rain rain and rain but we got it, it was hard and slippery the whole way but worth it. I can only recommend it.. we had only 10 minutes of open sky 😍 so beautiful!
- 5 Stars: Nature beauty at its finest. Took the doai route to the summit for 3hrs 9-25-22
- 4 Stars: Hiked Mt. Tanigawa (1977m) and was rewarded with stunning alpine scenery. The trail was rich with summer flowers, busy bees, and colorful butterflies. Though the weather was partly cloudy, the views of the surrounding green mountains were breathtaking. The ropeway ride offered a relaxing start, and the fresh mountain air made the climb refreshing. A beautiful destination for both nature lovers and hiking enthusiasts.

#### Top 10 Negative/Lowest Reviews:
No negative reviews available.


---
### Nagano Station
![Nagano Station](temp_images/ChIJh0bai5KGHWARO9I7pK11KTU.jpg)

- **Place ID:** `ChIJh0bai5KGHWARO9I7pK11KTU`
- **Address:** Kurita, Nagano, 380-0921, Japan
- **Overall Rating:** 4.1 ⭐
- **Review Summary:** {'text': 'Visitors consistently describe this station as clean, modern, and well-organized, offering a wide variety of shops, restaurants, and souvenir stores. Convenient parking options are available, including 30 minutes of free underground parking.\n\nSome reviews mention the service can be inefficient.', 'languageCode': 'en-US'}
- **Amenities Summary:** Parking: {"freeParkingLot": true, "paidParkingLot": true} | Accessibility: {"wheelchairAccessibleParking": true, "wheelchairAccessibleEntrance": true, "wheelchairAccessibleRestroom": true}
#### Top 10 Positive Reviews:
- 5 Stars: Very clean, lots or food options before we go onto our bullet train to Tokyo. There is a Becks Coffee shop for more Western coffees NewDays convenience store to pick up snacks.
- 5 Stars: Shinkansen stop on our way to Snow Monkey Park.  Station is easy to navigate.  For bus tickets to the park, hang a right after exiting the turnstiles, vending machines that dispense tickets will be to your right inside the ticket offices.  English instructions can be selected.  Not very intuitive but manageable.  There is a tourism office across from the big passageway where the staff were super helpful.  I highly recommend stopping here 1st.  They provided clearer and more information for us to get to Snow Monkey Park, like bus timetables and location of bus stop.  We used a locker as we had a small roller and a backpack coming from Tokyo.  Paid using Suica.  On our return, we lingered a bit at Nagano Station.  There are many places to eat.  We decided on a ramen place downstairs near the escalators.  There was also a supermarket with a fantastic Japanese prepared food selection to include local desserts, which I loved during my time in Japan.  We also grabbed some snacks from a convenience store (Family Mart or Lawson) at the main station level before taking the Shinkansen out.  There is also a mural commemorating Nagano as an Olympic city.  Overall very robust and functional station as Nagano benefited in terms of infrastructure from the Olympics.
- 5 Stars: Came for the Tateyama Kurobe Alpine route. Very helpful guide at the tourist information centre. You can buy tickets for the express bus to Ogizawa at the bus stop itself (Bus stop 25- East side). There are no reserved seats so no point buying early. We came 30 min early for the 750am bus mid November. Bus wasn't full and only 15 or so people on the bus. Very easy process.
- 4 Stars: A mid-size train station, quite typical of many stations in Japan. Clean, efficient, and well organized. Trains — including the Hokuriku Shinkansen — run very smoothly and on time.  The station is easy to navigate and less overwhelming than the major hubs like Tokyo or Shinjuku. It also has a decent selection of shops, restaurants, and souvenir stores inside and around the station.
- 4 Stars: The JR station is nice and modern. But the subway station is old and totally outdated without even IC card capability.

#### Top 10 Negative/Lowest Reviews:
No negative reviews available.


---
### Alpico Bus Ticket Office
![Alpico Bus Ticket Office](temp_images/ChIJ5SEdJ5OGHWARyjHm6TuPnSk.jpg)

- **Place ID:** `ChIJ5SEdJ5OGHWARyjHm6TuPnSk`
- **Address:** Suehirochō-1355-5 Minaminagano, Nagano, 380-0825, Japan
- **Overall Rating:** 3.5 ⭐
- **Review Summary:** {'text': 'Visitors report that this bus station offers tickets for various routes, accepts cash and other payment methods, and has staff who provide suitable advice.\n\nOther reviews mention bus scheduling information can be lacking and service can be inefficient.', 'languageCode': 'en-US'}
- **Amenities Summary:** Accessibility: {"wheelchairAccessibleParking": false, "wheelchairAccessibleEntrance": true}
#### Top 10 Positive Reviews:
- 4 Stars: If you want to explore the wonderful Togakushi Shrine area by bus, this is the starting point of your journey. Buy the ticket and get onboard the bus from this shop. Make sure to check the timetable in advance, and be aware that the bus did not had toilet inside.

#### Top 10 Negative/Lowest Reviews:
- 1 Stars: Lacking information that there are reserved buses and free seating buses. We though we where going to our scheduled activity in the morning. We ended up going a lot later. It was also not said the last scheduled bus going was also a reserved seating bus. When we got to our destination we learned the reserved seating bus going back was already full and we where forced to leave way early also. The end result we only had an hours and half for our activity. Only enough to get a pic. Not even slightly worth it. If only we could our time or money back.
- 1 Stars: We bought reserved return tickets on the highway bus for pick up at Togakushi Soba Museum but the bus didn’t stop even as we were waiting there early. We found a local bus but the Aloico Bus Company wouldn’t even give us a full refund. Very poor service after making us stressed and wasting our precious holiday time.
- 1 Stars: A lady told me not to buy a ticket for my ride back from Togakushi to Nagano on Saturday. She told me to take the last public (not Alpico) bus from Chusha at 17:20, this is actually the LAST transport ever during the day. Really? At 16:00 everything was already dead there. It was getting very cold. Iformation office in Chusha does not speak English. THERE IS NO TAXI IN TOGAKUSHI to take you back home. I decided not to risk with this last bus which would ride long after sunset when all the drivers would be gone, and started hitch-hiking. Luckily, 2 kind Japanese ladies picked up me and an Australian female traveler who had the same problem. I invite all the members of this company to Europe, so they could probably learn how this job has to be done.
- 1 Stars: Travel at own risk. Came on Saturday at 745am for 820am bus. There is already a super long queue.  Basically they want to sell tickets but don’t bother to account for the pax volume.  Make everyone queue up and wait in turn, based on their existing bus frequencies. No effort to increase load volume.


---
### Togakushi
![Togakushi](temp_images/ChIJp2dCGr4u9l8RWHLkTFFrG5s.jpg)

- **Place ID:** `ChIJp2dCGr4u9l8RWHLkTFFrG5s`
- **Address:** Togakushi, Nagano, 381-4101, Japan
- **Overall Rating:** N/A ⭐
#### Top 10 Positive Reviews:
No positive reviews available.

#### Top 10 Negative/Lowest Reviews:
No negative reviews available.


---
### Togakushi-Jinja Chusha
![Togakushi-Jinja Chusha](temp_images/ChIJqUaopGuF918R6vpw-DMDtyo.jpg)

- **Place ID:** `ChIJqUaopGuF918R6vpw-DMDtyo`
- **Address:** Chūsha-3506 Togakushi, Nagano, 381-4101, Japan
- **Overall Rating:** 4.4 ⭐
- **Review Summary:** {'text': 'Visitors say this shrine features impressive, centuries-old cedar trees and a serene atmosphere, with a small, beautiful waterfall on the grounds. They also highlight the convenient free parking and the unique omikuji fortune-telling experience where a priest recites prayers. Many appreciate the clean restrooms and the proximity to delicious soba restaurants.', 'languageCode': 'en-US'}
#### Top 10 Positive Reviews:
- 5 Stars: 2.Mar.2026: snow has melted and the path is slippery. **Right Togakushi Chusha middle temple, You can see very large cedar trees here. If you are looking for Togakushi Okusha, top high temple, with the famous path lined by two rows of cedar trees, you must walk about 2 km further from Chusha. Many visitors mistakenly think Chusha is Okusha. You can save nearly half the walking time by taking the bus from Kagami Pond instead ò Togakushi Chusha. However, opposite the bus stop at Togakushi Chusha there is a tourist information center where you can rent boots for snowy heavy condition if you walk from Chúha to Okusha.
- 5 Stars: We came during the snowy winter. For travelers coming via public transport, be prepared for a long and slippery hike. The forest and trail covered in snow are absolutely breathtaking, but the cold mountain can also, quite literally take your breath. We had a great time there and took quite some amazing photos along the way.
- 5 Stars: Supposedly one of Japan's holiest shrines, taking a walk down a path in the middle of a dense forest before being greeted by the stunning sight of massive cedar trees lining the sides was really relaxing.  Along the way, you'll also be treated to stunning snowy landscapes (if the weather's good) and the incredible sight of residue snow flying off tree tops as a cooling breeze blows through the forest. However, I wasn't able to reach the shrine itself as the path was blocked due to concerns of a possible avalanche.
- 4 Stars: Nice shrine, made even more special with the 700 year old ancient tree. So many amulets for sale but no English translation; and frowned upon when using phone and GoogleTranslate. No picture policy. I also saw a local getting a stamp in her souvenir book. She paid ¥1,000. I always thought those stamps were free.
- 4 Stars: Drive through the middle shrine. If walking from the lower shrine estimated time will be 1.30 -2 hours

#### Top 10 Negative/Lowest Reviews:
No negative reviews available.


---
### Togakushi Campsite
![Togakushi Campsite](temp_images/ChIJod13la8v9l8RPEMZJWsl3PI.jpg)

- **Place ID:** `ChIJod13la8v9l8RPEMZJWsl3PI`
- **Address:** Japan, 〒381-4101 Nagano, Togakushi, 大洞沢３６９４
- **Overall Rating:** 4.3 ⭐
- **Review Summary:** {'text': 'People say this campground offers spacious sites, clean restrooms with bidets, and well-maintained cooking areas with hot water. Visitors also highlight the wide variety of amenities, including a shop, coin showers, and a cafe, as well as the beautiful views of the Togakushi mountain range. Guests mention the staff are helpful and responsive, and the location is convenient for nearby attractions.', 'languageCode': 'en-US'}
- **Amenities Summary:** Parking: {"freeParkingLot": true, "paidParkingLot": true, "freeStreetParking": false, "paidStreetParking": false, "freeGarageParking": false, "paidGarageParking": false} | Accessibility: {"wheelchairAccessibleParking": true, "wheelchairAccessibleEntrance": true, "wheelchairAccessibleRestroom": true}
#### Top 10 Positive Reviews:
- 5 Stars: One of our best caming-experiences in Japan! For 1500 Yen per night with a camper you cant really go cheaper. The camping-ground itself is pretty big. We could park our camper wherever we wanted and had lots of space for us. Toilets and showers were superb, however a coin-exchange machine was missing for the 300 for the shower. However the shower itself was ideal! There was even a nice route around the area for a short jogg. All in all a wonderful experience! 10/10, would camp again
- 5 Stars: Lovely.with flgreat hiking courses around .farm coin shower.bbq and tent to rent
- 5 Stars: Beautiful site
- 5 Stars: Wonderful
- 5 Stars: 

#### Top 10 Negative/Lowest Reviews:
No negative reviews available.


---
### Upper Togakushi Shrine
![Upper Togakushi Shrine](temp_images/ChIJy80iekYu9l8RmSU5rs3-acA.jpg)

- **Place ID:** `ChIJy80iekYu9l8RmSU5rs3-acA`
- **Address:** 3506 Togakushi, Nagano, 381-4101, Japan
- **Overall Rating:** 4.6 ⭐
- **Amenities Summary:** Accessibility: {"wheelchairAccessibleParking": false, "wheelchairAccessibleEntrance": false}
#### Top 10 Positive Reviews:
- 5 Stars: Went there 31/12/2025. Need to walk from main road to this big tree. Icy condition. Snow shoes/spike would be great to have.  Walk is around 15-20 mins normal pace.
- 5 Stars: We trekked in with our snowshoes and had no issues getting to the shrine. We saw many others walking, which I would not recommend. The path becomes slippery and at the end, it is very easy to fall as you’re walking up or down from the shrine. Do not stand too close to the buildings as you just don’t know when the snow might slide from the top. The pathway up is beautiful thanks to the huge cedar trees. If you’re in the area don’t miss your chance!
- 5 Stars: Great time. I came there at the beginning of April and I only can walk pass the first Shrine, they blocked the way to second and third shrines due to bad road conditions (too much icy snow on the path). I do not know where to check this information, so wish you much more luck than us. If you travel with a group of at least 4 people, it’s better to take a taxi there cause the waiting time for a bus and travel time is quite long. Taxi costs around 12k. Worth visiting even though I could not complete the whole three shrines
- 5 Stars: We decided to hike the upper shrine and parked our car nearby. The trek took us about 40 mins each way or about 2km. Steep climbing of steps and bring a walking stick. The journey was wonderful and must say both my legs felt wobbly after 4km.
- 4 Stars: 4 out of 5, minus 1 only because it's basically a lovely walk through the forest and for the 1.5 hours from Tokyo plus the hour to the shrine at a cost of over $70, it wasn't worth it IMHO.  If one is in Nagano (for Obuse also, and other stuff, then a decent day trip, but I wouldn't just do Togakushi from Tokyo). By comparison, Nikko would be a way better day trip from Tokyo.  But it was nice and I'm glad I did it, but I don't see me ever doing this again.  Basically an lovely walk in the forest to see a very small shrine.  More than one way to get here but I used the private bus from Bus Stop #7  by Nagano Station (it's actually across the street fro the bus roundabout).  There is a bus ticket office right there, and I bought a round trip reserved seat ticket that left within 20 minutes.  There were quite a few people there already (and arriving after me), the bus was full.  This bus stops at six? 7 stops in the little village where all the shrines are, and also goes to a camping ground past this stop.  There are great maps for the area (in English), and shows you the hiking paths and distances/times.  You can get on the return bus from any of the stops.  I also had a reserved time for 2 PM but was done early and tired and was able to get on the 1230 bus. I just showed my return ticket and he gave me a new reserved seat. I do recommend buying a reserve seat for the return if you are day-tripping, your last bus might be sold out.  Once you leave the village (takes about 10-15 minutes to stop at all the stops) the bus doesn't stop for 30 minutes until it is back at Nagano station.  BTW, the last 10-15 minutes of hike to the shrine are stairs.  All the instagram photos shows the trees and path, but yeah, lots and lots of stairs, 5 stories worth (ish).  There is a restroom at the bus stop and a restaurant for soba and ice cream.  There are way more restaurants at the main shrine area. There is a bathroom along the trail.

#### Top 10 Negative/Lowest Reviews:
No negative reviews available.


---
### Togakushi Shrine Okusha (Main Shrine) Zuishinmon
![Togakushi Shrine Okusha (Main Shrine) Zuishinmon](temp_images/ChIJh-a7rVsu9l8R_A6PXgFs0hs.jpg)

- **Place ID:** `ChIJh-a7rVsu9l8R_A6PXgFs0hs`
- **Address:** Togakushi, Nagano, 381-4101, Japan
- **Overall Rating:** 4.6 ⭐
- **Review Summary:** {'text': 'Visitors say this shrine features a beautiful, historical red gate with a roof covered in lush plants, marking the entrance to a sacred area. They also highlight the serene atmosphere and the impressive, towering cedar trees lining the path. Guests mention the walk is peaceful and offers stunning natural scenery, making it a worthwhile experience in any season.', 'languageCode': 'en-US'}
- **Amenities Summary:** Accessibility: {"wheelchairAccessibleParking": false, "wheelchairAccessibleEntrance": false}
#### Top 10 Positive Reviews:
- 5 Stars: 随神门 (Zuishinmon Gate)  The Boundary: Human vs. Divine: It marks the exact point where the human world ends and the sacred realm of the gods begins.  Official Entrance: Passing through this red gate signifies you have officially left the "mortal world" behind.  The air cools noticeably the moment you step across the threshold.
- 5 Stars: My fiancé and I went in the winter so the upper shrine was closed unfortunately, but the cedar trees were well worth the visit. The towering lumber makes you really feel minuscule compared to nature. It was beautiful with the snow as well. I don’t think I’ve seen a more fitting combination. We ended up making a snow man to stand watch and warn future guests of the dangers that lie ahead.
- 5 Stars: This place is so magical in winter. Once in a lifetime experience. The cedar trees and snowy landscape completely displaying an out of this world scenery.  Be mindful of very very slippery road when you are on your way to the shrine, its no joke.
- 5 Stars: Such a beautiful shrine in the woods.  A steep stair walk up to the shrine and all part of the pilgrimage. If you’re carrying on up to the other shrines, though,  we might recommend starting above it and taking the trail down. Buy Alpico bus tickets in advance.
- 5 Stars: Absolutely stunning in winter. We went on 02/11/2025 and we arrived at around 9am and huge crowds came after us so it’s best to go a bit early. It’s around a 20min walk from the entrance to see the towering cedar trees. It was worth driving up here. Parking costs ¥800 for the first 3 hours.

#### Top 10 Negative/Lowest Reviews:
No negative reviews available.


---
### Togakushi Ninja Museum・Ninja Trick Mansion
![Togakushi Ninja Museum・Ninja Trick Mansion](temp_images/ChIJqb-wPfcu9l8Rl4esSWxms54.jpg)

- **Place ID:** `ChIJqb-wPfcu9l8Rl4esSWxms54`
- **Address:** 3688-12 Togakushi, Nagano, 381-4101, Japan
- **Overall Rating:** 4.4 ⭐
- **Review Summary:** {'text': 'Visitors say this ninja house offers a challenging and fun "karakuri yashiki" (trick house) experience with hidden doors and rotating walls, along with a thrilling "bikkuri-do" (shaking room) and shuriken throwing. They also highlight the reasonable admission fee and the friendly, helpful staff. People recommend visiting during less crowded times to fully enjoy the puzzles and avoid spoilers.', 'languageCode': 'en-US'}
#### Top 10 Positive Reviews:
- 5 Stars: this was so fun and worth the time and ticket price. we went here after visiting Togakushi Okusha and waiting for our Alpico bus. i won a stamp notepad after hitting 5/7 times at the shuriken dojo.  the ninja trick mansion is a must-go. it might take around 1 hour or so but very worth the try. it's so interesting, one hint is to try to move anything to find the way; going with a bigger group can make it easier. highly recommend for family and friends! i went on weekday so no wait and no queue at all.
- 5 Stars: After you come off of the bus from Nagano Station, cross the road immediately and climb the stairs, that's the museum location. It's absolutely a bargain, the ticket price compared to the experience that we had together as a family with young children. The highlights were definitely the ninja house, where you have to solve your way out, and also the ¥200 for 7 times throwing shuriken. You will even get a notepad for stamps if you hit it 5 times. The obstacles such as balancing beams and ropes were also fun and exhausting at the same time. Definitely a must if you have teenagers and children in your group. I recommend skipping the artifacts, the building was smelly.
- 5 Stars: Worth the cost of bus ride up there from nagano station, ninja house a def must especially for us big kids, then the walk through the cedar forest, but if your going to do the walk grab bus to ninja village then walk to shrine then down hill from there if your going to do that then def go early as and def buy a bear bell
- 5 Stars: Interesting museum full of old ninja tools. The ninja house is the most exciting part of this museum. So you enter this house and you have to find the exit just like how the ninjas were trained to find values and hidden doors (similar to an escape room experience). Another fun part was the shuriken throw, for ¥700 you get 7 shuriken and you have to hit the target. This was a fun place to visit and I'd definitely go back!
- 4 Stars: Brought my kids here to experience the life of a ninja on a Monday. No crowds at all, maybe around 3 families were here.  This place is not wheelchair friendly. However, can ask staff to open the side door instead going down the slope which is dangerous for wheelchair.  There's a cafe that sells delicious food. We enjoyed the rice with chicken which has smokey taste and ramen. Also has many cute items and lunch such as pizza, burger, udon, drinks, and desserts. Payment via cash or QR code. Free flow cold water, soup or hot green tea dispenser machine available.  Can buy admission only ticket or the all-inclusive passes. Also ninja costume rental is 500 yen per child and comes in different colours. We started from the adventure trail followed by indoor games. There were only 2 main staffs that assisted us in the indoor games.  The outdoor adventure trail has different levels. We accidentally tried the advance 5 star challenge which was abit scarry for us but managed to overcome as a family (youngest 6 years old) after a few tears due to fear of heights.  One of my child with special needs fell into the water during the "water walking spider" activity. The wooden float went faster than his hands that was holding the rope. Luckily we brought extra clothes to change. Managed to buy new socks and shoes from the souvenir shop. Credit card payment available at the shop.  Generally a memorable experience although it's smaller than the Uzumasa Kyoto Village Ninja escape room in Kyoto which also includes ninja performance and more indoor games for children.

#### Top 10 Negative/Lowest Reviews:
No negative reviews available.


---
### Uzuraya
![Uzuraya](temp_images/ChIJHZ_ui74u9l8Ry0pNBkGOHcA.jpg)

- **Place ID:** `ChIJHZ_ui74u9l8Ry0pNBkGOHcA`
- **Address:** 3229 Togakushi, Nagano, 381-4101, Japan
- **Overall Rating:** 4.5 ⭐
- **Review Summary:** {'text': "Diners like this soba restaurant's delicious, freshly made soba noodles and crispy tempura, especially the seasonal vegetables and sobagaki. They also highlight the warm, attentive, and professional service from the staff, who often offer helpful guidance. Guests mention the traditional, cozy atmosphere and appreciate the convenient system for waiting, which allows for nearby sightseeing.", 'languageCode': 'en-US'}
- **Amenities Summary:** Parking: {"freeParkingLot": true} | Accessibility: {"wheelchairAccessibleParking": false, "wheelchairAccessibleEntrance": false, "wheelchairAccessibleSeating": false}
#### Top 10 Positive Reviews:
- 5 Stars: The tempura was excellent!!! The best I’ve ever tasted- super thin coating, light and crispy. The services were so warm and cute. The staffs don’t speak much English but they tried their best to communicate and always smiling. The chef came out to talk to us about his Sake haha. Definitely coming back!!!!!! Soba size is a bit on the thinner size less chewy so that’s up to your preferences. They have this unique wasabi/ginger/peppery sauce that we love but they don’t sell when asked. If you guys know what it is please share.
- 5 Stars: Went on New Years (1st Jan), had a reservation at 1pm, and got in at around 2pm due to a huge crowd.  Soba Dumpling: (3/5) Big enough for 4 pax, soft texture and interesting taste (has no filling inside, just soba!) Served with miso wasabi sauce and other flavourings. It wasn’t our favourite but is interesting!  Hot Soba Soup: (5/5) Simple and delicious 😋  Zaru Soba (Cold): (5/5) I’ve only had a few sobas in Japan, but this is the best so far, being especially springy and smooth. It tasted so good we had to order more!
- 5 Stars: "I think this soba restaurant is the most popular in the Togakushi area, but they make a limited number of noodles each day. Here’s a tip: you should sign in early in the morning before you start visiting the shrines. You can sign up starting at 4 AM, and then you can come back to the restaurant whenever you want until 3 PM.
- 5 Stars: One of the best zaru soba place i visited in Japan. The soba noodle are freshly made here, soft and chewy super delicious. Great service with very friendly owner. Highly recommened.
- 5 Stars: Wonderful service staff who welcomed foreigners warmly in the magical and spiritual town of Togakushi.  Ordered the seasonal mountain vegetable tempura oroshi soba- what a feast of the season.  Texture of soba was thin and smooth, with a hint of buckwheat aroma.  Pro tip: long waiting time- leave your name with the staff once you arrive and start visiting the nearby temples and other attractions.

#### Top 10 Negative/Lowest Reviews:
No negative reviews available.


---
### Okusha-mae Naosuke
![Okusha-mae Naosuke](temp_images/ChIJNQfMB_cu9l8RmCY4-bhH1PE.jpg)

- **Place ID:** `ChIJNQfMB_cu9l8RmCY4-bhH1PE`
- **Address:** Japan, 〒381-4101 Nagano, Togakushi, 奥社3688-1
- **Overall Rating:** 4.3 ⭐
- **Review Summary:** {'text': "Diners like this restaurant's delicious soba, especially the cold soba, and the tempura is frequently described as crispy and flavorful. They also highlight the unique and tasty soba and kumazasa soft-serve ice cream, along with the spotless restrooms. Guests mention the staff are kind and attentive, and many appreciate the scenic counter and terrace seating.", 'languageCode': 'en-US'}
- **Amenities Summary:** Parking: {"paidParkingLot": true} | Accessibility: {"wheelchairAccessibleParking": false, "wheelchairAccessibleEntrance": false}
#### Top 10 Positive Reviews:
- 5 Stars: We only tried the warm oyaki from their side store. Vegetable, sweet walnut miso and red bean.  It was very tasty and I prefer the vegetable one. There are benches to sit on and enjoy the view as you relish the oyaki.
- 5 Stars: Nice staff and delicious food Recommend the 蓮根天婦羅 Lucky to have the window side seat Enjoy a pleasant lunch
- 4 Stars: Fairly priced soba compared to nearby restaurants. Also have the ability to just get tempura without the soba. Soba and tempura were good, not sure what the other reviews are about. Would recommend getting the shiso juice, unique but in a good way! Could be quite cramped due to seating being close together.

#### Top 10 Negative/Lowest Reviews:
- 2 Stars: Compared to other places, the price here was quite steep. Unfortunately, the food quality didn't justify the high cost. We waited in a long line to get in, but it turned out to be a letdown


---
### Matsumoto Backpackers
![Matsumoto Backpackers](temp_images/ChIJiyCxO4gOHWAR85d9ZL2b53I.jpg)

- **Place ID:** `ChIJiyCxO4gOHWAR85d9ZL2b53I`
- **Address:** 1-chōme-1-6 Shiraita, Matsumoto, Nagano 390-0863, Japan
- **Overall Rating:** 4.3 ⭐
- **Review Summary:** {'text': 'People say this hostel offers comfortable, well-maintained rooms with amenities like air conditioning, heating, and free Wi-Fi, and some rooms provide cooking utensils. Visitors also highlight the excellent value for money and the welcoming, social vibe. They also mention the staff are very friendly and accommodating, offering helpful local tips.', 'languageCode': 'en-US'}
- **Amenities Summary:** Accessibility: {"wheelchairAccessibleParking": false, "wheelchairAccessibleEntrance": false}
#### Top 10 Positive Reviews:
- 5 Stars: Would strongly recommend!  The place couldn’t have been tidier, safer, or more welcoming. The host Brian was lovely. He greeted me upon arrival with a smile (even though I was 45 minutes later than I said I would be), showed me around, and recommended locations in the area. He also helped direct me to Norikura where I was staying the next night - without him I would’ve been walking for 2+ hours.  Moreover, it’s located about 5 minutes walk from the station, making it awfully convenient.
- 5 Stars: Super convenient, quiet and well maintained. If you're lucky enough to bump into the owners during your stay, they are the kindest people in the world. Booked direct for the best deal.
- 5 Stars: The rooms were very comfortable and homely. The staff were incredibly friendly and welcoming. The location is perfect, just outside the busy hotel area of the city and adjacent to the river. It is also a 5 minute walk from the main shrine and a 10 minute walk from the castle. Would highly recommend this as high quality and very affordable accommodation for matsumoto.
- 5 Stars: I definitely recommend this place! We took family room at the second floor. It is neat complete with beddings air conditioner and heater and free wifi.  You can also bring food and cook since cooking utensils are provided.The place is just a short walk from the train station. The owner is also very accommodating, he even gave us tips on the places we have to visit.

#### Top 10 Negative/Lowest Reviews:
- 2 Stars: There is absolutely no reception space, anybody can come in and out, no security, the door code is useless. No one available in case of questions about the surroundings or possible issues around the house. Tried to use the heating in the bedroom and a lot of smoke came out, very dangerous with tatami floors. Absolutely no isolation from outside or through the walls, you can hear everyone's business in the bathroom, kitchen, very small communal room. Disappointing


---
### Matsumoto Castle
![Matsumoto Castle](temp_images/ChIJmVmaCoUOHWARVPa8-iAOLZA.jpg)

- **Place ID:** `ChIJmVmaCoUOHWARVPa8-iAOLZA`
- **Address:** 4-1 Marunouchi, Matsumoto, Nagano 390-0873, Japan
- **Overall Rating:** 4.5 ⭐
- **Review Summary:** {'text': 'People say this castle is beautiful, with a striking black exterior, and features interesting historical displays, including a collection of firearms. Visitors also highlight the well-maintained grounds, which are perfect for photography, and the option to purchase e-tickets for smoother entry. They also mention the staff are friendly and helpful, with some offering free English-speaking tours.', 'languageCode': 'en-US'}
- **Amenities Summary:** Parking: {"freeParkingLot": true, "paidParkingLot": true, "paidStreetParking": true} | Accessibility: {"wheelchairAccessibleParking": false, "wheelchairAccessibleEntrance": false}
#### Top 10 Positive Reviews:
- 5 Stars: Unlike many other rebuilt castles in Japan, this is an original castle, one of Japan's twelve original keeps. You can really feel the age and authenticity as you climb through the interior.  The exterior and the surrounding moat are impeccably kept and look very classy.  The wooden stairs inside are incredibly steep and narrow (almost like ladders in some spots). You’ll need to take your shoes off at the entrance, so wear nice socks!  They have free English-speaking volunteer guides, they are very friendly and knowledgeable.  The night illumination and light projection are a must-see. The reflection of the castle in the moat during the show is perfect for photos.
- 5 Stars: The exterior of the castle was my highlight along with the staff. The grounds were beautiful and I have a huge appreciation for the beautiful structure that is 3x older then our country! Truly impressive. We only just stumbled across this castle and last minute decided to stop as we were passing through and so pleased we did! We ended up buying a ticket and entering the castle - line was minimal but as we left we could see the line growing, so possibly worth purchasing online and getting there early.
- 5 Stars: Matsumoto Castle was an absolutely wonderful experience. We were lucky to visit on a stunning day with relatively light crowds, which made it even more enjoyable to take in the history and atmosphere of the castle.  Exploring inside really gives you a sense of its past, though be prepared — some of the stairs are quite steep and may be challenging for those with mobility issues.  The cherry blossoms were just beginning to bloom, adding a beautiful touch to the setting. One of my favorite moments was seeing all the koi fish gliding through the water around the castle — it made the whole scene feel especially peaceful and picturesque.  Highly recommend visiting if you’re in the area!
- 5 Stars: We stopped by Matsumoto Castle and even just seeing it from the outside was impressive. The castle looks stunning and striking against the surroundings and definitely has that classic, majestic feel.  There are quite a few parking lots around the area, so it’s easy enough to find a spot and just walk over. There are also clean public toilets right outside, which is convenient. We didn’t go inside this time, but honestly, even just taking in the view from the outside felt worth it. If you’re in the area, it’s an easy and worthwhile stop.
- 4 Stars: Beautiful Castle, But Not for Everyone  Matsumoto Castle is undoubtedly one of the most beautiful castles in Japan. The striking black exterior, the surrounding grounds, and the reflections in the moat make it a wonderful place to visit and photograph. The castle has a genuine historic feel and is far more authentic than many other castles that have been heavily reconstructed.  However, visitors should be aware that the interior is not suitable for everyone. The wooden staircases inside are extraordinarily steep, narrow, and, at times, quite claustrophobic. Going up is challenging enough, but coming down requires considerable care. A slip could easily result in a serious fall, particularly for older visitors or anyone with mobility issues.  That said, the steep staircases are part of the castle’s original character and history, so they do provide an authentic glimpse into how these structures were built centuries ago.  Overall, a fascinating and beautiful historic site that is well worth visiting, but it is important to know what to expect before climbing to the upper levels.

#### Top 10 Negative/Lowest Reviews:
No negative reviews available.


---
### Nawate Shopping Street
![Nawate Shopping Street](temp_images/ChIJR-XzQo4OHWAR3o9GleBXoi0.jpg)

- **Place ID:** `ChIJR-XzQo4OHWAR3o9GleBXoi0`
- **Address:** Japan, 〒390-0874 Nagano, Matsumoto, Ōte, 3-chōme−4−３ 松本Ｍ―１
- **Overall Rating:** 4 ⭐
- **Amenities Summary:** Parking: {"paidParkingLot": true} | Accessibility: {"wheelchairAccessibleParking": false}
#### Top 10 Positive Reviews:
- 5 Stars: A charming little street on the way to Matsumoto Castle. Nawate Street is short but packed with character. I loved the scenic riverside walk with beautiful vintage lamp posts. It’s right next to Yohashira Shrine, where you can sit and enjoy a peaceful moment. Great local snacks and lovely small shops. Highly recommend for a relaxing break!"
- 4 Stars: Came here around first week of February 2026 at night time but there were no open stores, not sure why, thought I’ll be seeing food stalls or yatais. It’s the gateway towards Matsumoto castle and the views during my walk from Matsumoto station to here was pretty neat.
- 4 Stars: I think if you’re walking to the castle it’s worth the detour down here, there were a few shops selling a variety of things. It was very quiet and there’s a few cafes at the end. Wasn’t exactly a life changing experience but cute.
- 4 Stars: Long shopping street but it is not so happening. Slow and relaxing stroll with many shops selling food and sounviners. There is a big frog statue for photographing, and a frog shrine too
- 4 Stars: I visited this area in mid-July 2023. There are plenty of souvenir shops, cafes, and snack stalls around. Right at the entrance, you’re welcomed by a frog statue—super cute and unique! I really loved the retro vibe of the whole place. If you walk further in, there are lots of charming little alleys to explore. Definitely worth taking your time to wander around!

#### Top 10 Negative/Lowest Reviews:
No negative reviews available.


---
### Nakamachi Dori
![Nakamachi Dori](temp_images/ChIJURknZY4OHWARiO7O3rM2h4k.jpg)

- **Place ID:** `ChIJURknZY4OHWARiO7O3rM2h4k`
- **Address:** Nakamachi Dori, Chūō, Matsumoto, Nagano 390-0811, Japan
- **Overall Rating:** N/A ⭐
#### Top 10 Positive Reviews:
No positive reviews available.

#### Top 10 Negative/Lowest Reviews:
No negative reviews available.


---
### Matsumoto Karaage Center
![Matsumoto Karaage Center](temp_images/ChIJ25LHw4sOHWARGO_jEMazFKk.jpg)

- **Place ID:** `ChIJ25LHw4sOHWARGO_jEMazFKk`
- **Address:** Japan, 〒390-0815 Nagano, Matsumoto, Fukashi, 1-chōme−1−１ MIDORI 4階
- **Overall Rating:** 3.7 ⭐
- **Review Summary:** {'text': "Diners like this restaurant's crispy and juicy karaage and sanzoku-yaki, which are generously portioned and flavorful. They also highlight the convenient location within the train station and the reasonable prices.\n\nOther reviews mention the service can be inattentive.", 'languageCode': 'en-US'}
- **Amenities Summary:** Parking: {"paidParkingLot": true} | Accessibility: {"wheelchairAccessibleParking": false, "wheelchairAccessibleEntrance": false, "wheelchairAccessibleSeating": false}
#### Top 10 Positive Reviews:
- 5 Stars: Amazing fried chicken! I’m genuinely confused by the low ratings here. The chicken is perfectly fried—crispy, juicy, and not dry at all. They have 3 varieties of sauces to choose from and great side dishes to balance out the grease. Ordering via QR code is super convenient, and the service is lightning fast. Highly recommended!
- 5 Stars: This was recommended by the Lonely Planet- Japan book, and definitely worth the recommendation. They had a variety of food sets at very affordable prices. The chicken is so tender juicy and crunchy. They also offer takeout.
- 5 Stars: The two-style chicken menus were delicious, offering a tasty flavor and crispy texture while keeping the meat juicy. Various sauces were available on the table. Additionally, the all-you-can-eat pickled bean sprouts were very yummy. However, when it comes to the service, it was a little uncomfortable. I arrived there around 6:30 PM and joined the queue around 7 PM. While I was eating, I felt like the staff really wanted to go home because they came and looked at me many times, and not just one staff, but all of them. They did this to many tables around me as well.
- 4 Stars: Upon arrival at Matsumoto station at around lunchtime, our group of five excitedly hurried to the fourth floor of the Midori mall to eat at the interestingly-named Matusmoto Karaage Center to try the local specialty of sanzoku-yaki. We saw a short line that was moving fast and wrote our names down on a queuing list outside the entrance and were guided to our table within 10-15 minutes. The interior was painted in black with a seating section that required you to remove your shoes, with a slightly noisy interior playing music and with slightly loud conversations. Japanese and English menus are available.  We ordered the sanzoku-yaki curry, double chicken (katsu and sanzoku-yaki) with tartar sauce and apple sauce, and regular sanzoku--yaki. Our food was served within 10-15 minutes. The highlight of this restaurant, the sanzoku-yaki, was larger than the average karaage and resembled a katsu in size, being very crispy, juicy, and tasty, bearing a hint of garlic in its inherent flavor - giving a unique experience to those looking for a new way to enjoy karaage. The curry was standard and more on the sweet side, not being too spicy and was complimented by fukujinzuke on the side. The double chicken was served with a small portion of chicken katsu, which was standardly juicy, crispy, and tasty. However, we found that the mustard-based dipping sauce served with the other sanzoku-yaki set was tastier than either the tartar and apple sauce and thus paled in comparison. In general, the portions were large and reasonably filled our stomachs prior to exploring Matsumoto. We spent about 1000-2000 JPY per person.  Overall, I would recommend this place for karaage enthusiasts looking for a large meal with a short wait and reasonable prices.
- 4 Stars: Conveniently located on the 4th floor of the Matsumoto Train Station. Menu has only a few options, but whatever is there is great! The karaage is fantastic - batter is light and cripsy and the chicken is sooo tender and juicy. And the rice was really fluffy and addictive!  Best to arrive really early because of limited seating and also long queues due to popularity. They close at 8 sharp; last order is at 7.30 and payment has to be made by 7.50pm - so it can be rushed for latecomers. They do takeaways too.

#### Top 10 Negative/Lowest Reviews:
No negative reviews available.


---
### Shinshimashima Station
![Shinshimashima Station](temp_images/ChIJr1vJiiIUHWARybTUdPnBmEM.jpg)

- **Place ID:** `ChIJr1vJiiIUHWARybTUdPnBmEM`
- **Address:** Hata, Matsumoto, Nagano 390-1401, Japan
- **Overall Rating:** 4.1 ⭐
- **Amenities Summary:** Accessibility: {"wheelchairAccessibleParking": true, "wheelchairAccessibleEntrance": true, "wheelchairAccessibleRestroom": true}
#### Top 10 Positive Reviews:
- 5 Stars: A small station that serves as the junction between Kamikochi and Matsumoto. There’s a small waiting area where you can stay until just before your train arrives.
- 5 Stars: A very small train station at the end of the line.  A jumping off spot to Kamikochi through bus transfer.  Very clean restroom and a sizable waiting room with lots of vending machines for drinks but I don't see food vendors.
- 5 Stars: A small station with toilets, coin lockers, and vending machines. Signs were easy to understand. Also, the staff were really helpful.
- 5 Stars: Bus to Kamikochi and a few other locations from here. Credit cards are accepted. Quiet and peaceful. The staff is helpful and will guide you well.
- 4 Stars: A small terminal station of Matsumoto Dentetsu Kamikochi line. It's also a bus terminal for travelling to Kamikochi. There are some basic facilities such as toilets and vending machines.

#### Top 10 Negative/Lowest Reviews:
No negative reviews available.


---



Saved advanced reviews to outputs/japan_place_reviews.md


In [18]:
place_name = places_to_search[0]
for place_name in places_to_search:
    params = {
        "query": f"{place_name} Japan",
        "key": MAPS_API_KEY
    }
    res = requests.get(search_url, params=params).json()

    if res.get("status") == "OK" and res.get("results"):
        place = res["results"][0]
place

{'business_status': 'OPERATIONAL',
 'formatted_address': '2 Chome-2 Fukuromachi, Kanazawa, Ishikawa 920-0909, Japan',
 'geometry': {'location': {'lat': 36.5720575, 'lng': 136.6573685},
  'viewport': {'northeast': {'lat': 36.57330393029149,
    'lng': 136.6586789802915},
   'southwest': {'lat': 36.5706059697085, 'lng': 136.6559810197085}}},
 'icon': 'https://maps.gstatic.com/mapfiles/place_api/icons/v1/png_71/lodging-71.png',
 'icon_background_color': '#909CE1',
 'icon_mask_base_uri': 'https://maps.gstatic.com/mapfiles/place_api/icons/v2/hotel_pinlet',
 'name': 'Soki Kanazawa',
 'photos': [{'height': 4730,
   'html_attributions': ['<a href="https://maps.google.com/maps/contrib/117482371150091561694">SOKI KANAZAWA</a>'],
   'photo_reference': 'AaVGc3klyNncY3EYm2w05yG4NNLpl-n9DsyOEC17cc3DmqVgCdr6smpRhGUT8VjJUtRCCqhwIYY-YnFnILpHJHX-khdHxziKMSB37VmyZ1fMb3TNZDlJ2hh10rrP5pel4xqQryiWIULW0Q7PPsKOwEzebaIJbhoPmSOtZzVigocYIMjMjhthEEdxgAskGoIYySwi0YGc2an2efLAxfcpihBxUN6EuBzaV6SjW4MPvlbVQaxYLJQC6n28

In [ ]:
details_params = {
    "place_id": 'ChIJsy0pbliNGGARUMTvdy0TrsE',
    "fields": "reviews,website",
    "key": MAPS_API_KEY
    }
det_res = requests.get(details_url, params=details_params).json()

reviews_text = ""
review_summary = "No reviews available to summarize."

if det_res.get("status") == "OK":
    result = det_res.get("result", {})

In [41]:
len(result['reviews'])

# [0]['rating'] 

5

In [42]:
for i in range(len(result['reviews'])):
  result['reviews'] if result['reviews'][i]['rating'] < 3 else '' 